In [50]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
# from sklearn.preprocessing import

import lightgbm as lgbm

import gurobipy as gp
from gurobipy import GRB

import lightgbm as lgbm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Layer, Concatenate
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import EarlyStopping

import torch
import torch.nn as nn
import torch.nn.functional as F

import gurobipy as gp
from gurobipy import GRB

# Optional: torch_geometric for graph layers
try:
    from torch_geometric.nn import GATConv
    from torch_geometric.data import Data as GeoData
except Exception:
    GATConv = None
    GeoData = None


2026-03-21 22:25:57.115638: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
data_23_24 = pd.read_csv('https://raw.githubusercontent.com/ilyandho/FPL-Optimal-Transfer/refs/heads/FantasyGo/FPL%20predictors/Fantasy%20Go/fantasyGo_FPL.csv')
fg_managers = pd.read_csv('https://raw.githubusercontent.com/ilyandho/FPL-Optimal-Transfer/refs/heads/FantasyGo/FPL%20predictors/Fantasy%20Go/fgo_managers.csv')
fg_managers['points'] = fg_managers['points'].str.split(' ').str[0]
fg_managers['points'] = pd.to_numeric(fg_managers['points'], errors='coerce')

In [78]:
data_23_24 = pd.read_csv('../FPL predictors/with new features/data/joint/23-24/merged_player_data.csv')
fg_managers = pd.read_csv('https://raw.githubusercontent.com/ilyandho/FPL-Optimal-Transfer/refs/heads/FantasyGo/FPL%20predictors/Fantasy%20Go/fgo_managers.csv')
fg_managers['points'] = fg_managers['points'].str.split(' ').str[0]
fg_managers['points'] = pd.to_numeric(fg_managers['points'], errors='coerce')

## Data Prep


In [4]:
# play_data[['was_home', 'position', 'team', 'opponent_team']]

In [5]:
play_data = data_23_24.copy()

play_data['team'] = play_data['player_team']
play_data['opponent_team'] = play_data.apply(lambda row: row['a_team'] if row['was_home'] == 1 else row['h_team'], axis=1)

# play_data.columns.tolist()

In [6]:
v = [
    'understat_id', 'player_team', 'FPL_Name', 'Understat_Name', 'position', 'team', 'opponent_team', 'fpl_name',  'h_team', 'a_team', 'fpl_id', 'round', 'was_home',
    'date', 'team_h_difficulty', 'team_a_difficulty',
    'assists_x', 'clean_sheets', 'creativity', 'element', 'expected_assists', 'expected_goal_involvements', 'expected_goals', 'expected_goals_conceded',
    'goals_conceded', 'goals_scored', 'ict_index', 'influence',  'minutes', 'own_goals', 'penalties_missed', 'penalties_saved', 'red_cards', 'saves', 'selected',
    'starts', 'team_a_score', 'team_h_score', 'threat', 'transfers_balance', 'transfers_in', 'transfers_out', 'value',  'yellow_cards', 'goals', 'shots', 'xG',
    'h_goals', 'a_goals','xA', 'key_passes', 'npg', 'npxG', 'xGChain', 'xGBuildup',  'xP', 'pts_bps',
 ]

rolling_data = play_data[v].rename(columns={'assists_x': 'assists'}, inplace=False)
rolling_data

## Cater for the mssing rounds before rolling
# 1. Get all unique players and all unique rounds
all_players = rolling_data['fpl_id'].unique()
all_rounds = range(rolling_data['round'].min(), rolling_data['round'].max() + 1)

# 2. Create a MultiIndex of every player x every round
multi_idx = pd.MultiIndex.from_product([all_players, all_rounds], names=['fpl_id', 'round'])

# 3. Reindex the dataframe
# This inserts "empty" rows for missing player/round combinations
rolling_data = rolling_data.set_index(['fpl_id', 'round']).reindex(multi_idx).reset_index()

# Fill statistical columns with 0
# stat_columns = ['goals_scored', 'goals_conceded', 'expected_goals_conceded']
rolling_data = rolling_data.fillna(0)

# Sort to ensure chronological order for the rolling window
rolling_data = rolling_data.sort_values(['fpl_id', 'round'])

rolling_data

,fpl_id,round,understat_id,player_team,FPL_Name,Understat_Name,position,team,opponent_team,fpl_name,...,h_goals,a_goals,xA,key_passes,npg,npxG,xGChain,xGBuildup,xP,pts_bps
4978,2,1,0.0,0,0,0,0,0,0,0,...,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0
4979,2,2,0.0,0,0,0,0,0,0,0,...,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0
4980,2,3,0.0,0,0,0,0,0,0,0,...,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0
4981,2,4,0.0,0,0,0,0,0,0,0,...,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0
4982,2,5,0.0,0,0,0,0,0,0,0,...,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14321,866,34,0.0,0,0,0,0,0,0,0,...,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0
14322,866,35,0.0,0,0,0,0,0,0,0,...,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0
14323,866,36,0.0,0,0,0,0,0,0,0,...,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0
14324,866,37,0.0,0,0,0,0,0,0,0,...,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0


In [7]:
# 1. Calculate the maximum 'selected' value across all players for each round
max_per_round = rolling_data.groupby('round')['selected'].max()

# 2. Shift the values down by 1 round (so Round 2 gets the max of Round 1, etc.)
prev_round_max = max_per_round.shift(1)
prev_round_total = np.floor(prev_round_max/0.80)
prev_round_total
# # 3. Map these 'previous round max' values back to the original DataFrame
rolling_data['total_managers'] = rolling_data['round'].map(prev_round_total)
rolling_data['total_managers']

rolling_data['selected_by_percent'] = (rolling_data['selected'] / rolling_data['total_managers'])
rolling_data['ownership_change'] = rolling_data.groupby('fpl_id')['selected'].diff()
rolling_data['percentage_net_transfers'] = rolling_data.groupby('fpl_id')['selected_by_percent'].diff()

In [8]:
season = "2324" # Change this for historical seasons
url = f"https://www.football-data.co.uk/mmz4281/{season}/E0.csv"

# It's good practice to use a custom User-Agent to avoid blocks
odds_data = pd.read_csv(url, low_memory=False)

# 1. Clean up dates in betting data
odds_data['Date'] = pd.to_datetime(odds_data['Date'], dayfirst=True).dt.date

# 2. Extract key columns (B365H = Bet365 Home Odds, B365D = Draw, B365A = Away)
odds_subset = odds_data[['Date', 'HomeTeam', 'AwayTeam', 'B365H', 'B365D', 'B365A']].copy()
odds_subset = odds_subset.rename(columns={'Date': 'date'})

def add_odds(row):
    win = round(1/row['B365H'], 5)
    draw = round(1/row['B365D'], 5)
    lose = round(1/row['B365A'], 5)

    # Normalize the probabilities (to make the probabilities sum to 100%)
    sum_percent = win + draw + lose
    win_prob = round(win/sum_percent, 3)
    draw_prob = round(draw/sum_percent, 3)
    lose_prob = round(lose/sum_percent, 3)

    return pd.Series([win_prob, draw_prob, lose_prob])

odds_subset[['win_prob', 'draw_prob', 'lose_prob']] = odds_subset.apply(add_odds, axis=1)
odds_subset

,date,HomeTeam,AwayTeam,B365H,B365D,B365A,win_prob,draw_prob,lose_prob
0,2023-08-11,Burnley,Man City,8.00,5.50,1.33,0.118,0.172,0.710
1,2023-08-12,Arsenal,Nott'm Forest,1.18,7.00,15.00,0.802,0.135,0.063
2,2023-08-12,Bournemouth,West Ham,2.70,3.40,2.55,0.351,0.278,0.371
3,2023-08-12,Brighton,Luton,1.33,5.50,9.00,0.720,0.174,0.106
4,2023-08-12,Everton,Fulham,2.20,3.40,3.30,0.432,0.280,0.288
...,...,...,...,...,...,...,...,...,...
375,2024-05-19,Crystal Palace,Aston Villa,1.85,4.33,3.60,0.515,0.220,0.265
376,2024-05-19,Liverpool,Wolves,1.17,8.00,15.00,0.817,0.119,0.064
377,2024-05-19,Luton,Fulham,2.90,3.90,2.20,0.327,0.243,0.431
378,2024-05-19,Man City,West Ham,1.08,12.00,21.00,0.876,0.079,0.045


In [9]:
# Make sure the names match in the player_data df and odds_subset df
team_map = {
    'Burnley' : 'Burnley',
    'Arsenal' : 'Arsenal',
    'Bournemouth' : 'Bournemouth',
    'Brighton' : 'Brighton and Hove Albion',
    'Everton' : 'Everton',
    'Sheffield United' : 'Sheffield United',
    'Newcastle' : 'Newcastle United',
    'Brentford' : 'Brentford',
    'Chelsea' : 'Chelsea',
    'Man United' : 'Manchester United',
    "Nott'm Forest" : "Nottingham Forest",
    'Fulham' : 'Fulham',
    'Liverpool' : 'Liverpool',
    'Wolves' : 'Wolverhampton Wanderers',
    'Tottenham' : 'Tottenham Hotspur',
    'Man City' : 'Manchester City',
    'Aston Villa' : 'Aston Villa',
    'West Ham' : 'West Ham United',
    'Crystal Palace' : 'Crystal Palace',
    'Luton' : 'Luton'
}


odds_subset['HomeTeam'] = odds_subset['HomeTeam'].map(team_map)
odds_subset['AwayTeam'] = odds_subset['AwayTeam'].map(team_map)


odds_melted = odds_subset.melt(
    id_vars=['date','B365H', 'B365D', 'B365A', 'win_prob', 'draw_prob', 'lose_prob'],
    value_vars=['HomeTeam', 'AwayTeam'],
    var_name='side',
    value_name='team'
)

odds_melted['date'] = odds_melted['date'].astype(str)

player_data_odds = rolling_data.merge(odds_melted[['date', 'win_prob', 'draw_prob', 'lose_prob','team']],
                  on=['date', 'team'],
                  how='left'
                  )


In [10]:

player_data_odds[['round', 'team_h_difficulty', 'team_a_difficulty', 'team', 'opponent_team']]


,round,team_h_difficulty,team_a_difficulty,team,opponent_team
0,1,0.0,0.0,0,0
1,2,0.0,0.0,0,0
2,3,0.0,0.0,0,0
3,4,0.0,0.0,0,0
4,5,0.0,0.0,0,0
...,...,...,...,...,...
25341,34,0.0,0.0,0,0
25342,35,0.0,0.0,0,0
25343,36,0.0,0.0,0,0
25344,37,0.0,0.0,0,0


In [11]:
rolling_features = [
    'assists', 'clean_sheets', 'creativity', 'element', 'expected_assists', 'expected_goal_involvements', 'expected_goals', 'expected_goals_conceded', 'team_a_difficulty',
    'team_h_difficulty',
    'goals_conceded', 'goals_scored', 'ict_index', 'influence',  'minutes', 'own_goals', 'penalties_missed', 'penalties_saved', 'red_cards', 'saves', 'selected',
    'starts', 'team_a_score', 'team_h_score', 'threat', 'transfers_balance', 'transfers_in', 'transfers_out', 'value',  'yellow_cards', 'goals', 'shots', 'xG',
    'h_goals', 'a_goals','xA', 'key_passes', 'npg', 'npxG', 'xGChain', 'xGBuildup',  'xP', 'pts_bps', 'selected_by_percent', 'ownership_change', 'percentage_net_transfers'
]

SPANS = [1,3,5]

# Sort to ensure chronological order for the rolling window
data_to_roll = player_data_odds.sort_values(['fpl_id', 'round']).reset_index(drop=True)


### Simple Averages


In [12]:
data_to_roll = data_to_roll[data_to_roll['team'] != 'Mainz 05']
# Averagae values
new_features = {}
for col in rolling_features:
    # The rolling window now spans actual calendar rounds
    grp = data_to_roll.groupby('fpl_id')[col]

    for span in SPANS:
        new_features[f'{col}_rolling_{span}'] = grp.transform(
            lambda x: x.shift(1).rolling(span, min_periods=span).sum()/span
        )

# Join all new columns at once (no fragmentation)
data_to_roll = pd.concat(
    [data_to_roll, pd.DataFrame(new_features)],
    axis=1
)

data_to_roll

,fpl_id,round,understat_id,player_team,FPL_Name,Understat_Name,position,team,opponent_team,fpl_name,...,pts_bps_rolling_5,selected_by_percent_rolling_1,selected_by_percent_rolling_3,selected_by_percent_rolling_5,ownership_change_rolling_1,ownership_change_rolling_3,ownership_change_rolling_5,percentage_net_transfers_rolling_1,percentage_net_transfers_rolling_3,percentage_net_transfers_rolling_5
0,2,1,0.0,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,2,0.0,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,3,0.0,0,0,0,0,0,0,0,...,NaN,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
3,2,4,0.0,0,0,0,0,0,0,0,...,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN
4,2,5,0.0,0,0,0,0,0,0,0,...,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25341,866,34,0.0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25342,866,35,0.0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25343,866,36,0.0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25344,866,37,0.0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Weighted Averages


In [13]:
ewm_features = {}
grouped = data_to_roll.groupby('fpl_id', sort=False)

# for col  in EWMA_SPANS:
for col in rolling_features:
    shifted = grouped[col].shift(1)
    for span in SPANS:
        ewm_features[f'{col}_{span}_ewm'] = (
            shifted.groupby(data_to_roll['fpl_id'])
            .ewm(span=span, adjust=False)
            .mean()
            .reset_index(level=0, drop=True)
        )

data_to_roll = pd.concat([data_to_roll, pd.DataFrame(ewm_features)], axis=1)

data_to_roll = data_to_roll[data_to_roll['FPL_Name'] != 0]
data_to_roll.to_csv('./rolled_data_23_24.csv', index=False)

In [14]:
cols = [
    'fpl_id', 'round', 'fpl_name', 'position', # 'team_id', # 'team',
    'opponent_team', 'xP', # 'was_home',
    'selected_by_percent','value', 'transfers_in', 'transfers_out',  'selected', 'ownership_change', #'percenatge_net_transfers',
    # 'cbitr_volume_90', 'cbitr_ewm_90', 'cbitr_ewm_probability',  'cbitr_probability','elo',
    # 'team_h_difficulty', 'team_a_difficulty',
    # 'win_prob', 'draw_prob', 'lose_prob', 'anytime_logit', 'two_plus_logit', 'hatrick_logit',

    'creativity_rolling_1', 'creativity_rolling_3', 'creativity_rolling_5', 'influence_rolling_1', 'influence_rolling_3', 'influence_rolling_5', 'threat_rolling_1',
    'threat_rolling_3', 'threat_rolling_5', 'minutes_rolling_1', 'minutes_rolling_3', 'minutes_rolling_5', 'pts_bps_rolling_1', 'pts_bps_rolling_3',
    'pts_bps_rolling_5', 'expected_goals_rolling_1', 'expected_goals_rolling_3', 'expected_goals_rolling_5', 'expected_assists_rolling_1',
    'expected_assists_rolling_3', 'expected_assists_rolling_5', 'xP_rolling_1', 'xP_rolling_3', 'xP_rolling_5', 'expected_goals_conceded_rolling_1',
    'expected_goals_conceded_rolling_3', 'expected_goals_conceded_rolling_5', 'goals_conceded_rolling_1', 'goals_conceded_rolling_3', 'goals_conceded_rolling_5',
    'goals_scored_rolling_1', 'goals_scored_rolling_3', 'goals_scored_rolling_5', 'shots_rolling_1', 'shots_rolling_3', 'shots_rolling_5', 'key_passes_rolling_1',
    'key_passes_rolling_3', 'key_passes_rolling_5', 'npg_rolling_1', 'npg_rolling_3', 'npg_rolling_5', 'npxG_rolling_1', 'npxG_rolling_3', 'npxG_rolling_5',
    'goals_rolling_1', 'goals_rolling_3', 'goals_rolling_5', 'xG_rolling_1', 'xG_rolling_3', 'xG_rolling_5', 'xA_rolling_1', 'xA_rolling_3', 'xA_rolling_5',
    'saves_rolling_1', 'saves_rolling_3', 'saves_rolling_5', 'starts_rolling_1', 'starts_rolling_3', 'starts_rolling_5', 'yellow_cards_rolling_1',
    'yellow_cards_rolling_3', 'yellow_cards_rolling_5', 'red_cards_rolling_1', 'red_cards_rolling_3', 'red_cards_rolling_5', 'assists_rolling_1', 'assists_rolling_3',
    'assists_rolling_5', 'clean_sheets_rolling_1', 'clean_sheets_rolling_3', 'clean_sheets_rolling_5', 'value_rolling_1', 'value_rolling_3', 'value_rolling_5',
    'ict_index_rolling_1', 'ict_index_rolling_3', 'ict_index_rolling_5', 'selected_rolling_1', 'selected_rolling_3', 'selected_rolling_5', 'transfers_in_rolling_1',
    'transfers_in_rolling_3', 'transfers_in_rolling_5', 'transfers_out_rolling_1', 'transfers_out_rolling_3', 'transfers_out_rolling_5',
    'xGChain_rolling_1', 'xGChain_rolling_3', 'xGChain_rolling_5', 'xGBuildup_rolling_1', 'xGBuildup_rolling_3',
    'xGBuildup_rolling_5', 'expected_goal_involvements_rolling_1', 'expected_goal_involvements_rolling_3', 'expected_goal_involvements_rolling_5',
    'selected_by_percent_rolling_1', 'selected_by_percent_rolling_3', 'selected_by_percent_rolling_5',
    'creativity_1_ewm', 'creativity_3_ewm', 'creativity_5_ewm', 'influence_1_ewm', 'influence_3_ewm', 'influence_5_ewm', 'threat_1_ewm', 'threat_3_ewm', 'threat_5_ewm',
    'minutes_1_ewm', 'minutes_3_ewm', 'minutes_5_ewm', 'pts_bps_1_ewm', 'pts_bps_3_ewm', 'pts_bps_5_ewm', 'expected_goals_1_ewm', 'expected_goals_3_ewm',
    'expected_goals_5_ewm', 'expected_assists_1_ewm', 'expected_assists_3_ewm', 'expected_assists_5_ewm', 'xP_1_ewm', 'xP_3_ewm', 'xP_5_ewm',
    'expected_goals_conceded_1_ewm', 'expected_goals_conceded_3_ewm', 'expected_goals_conceded_5_ewm', 'goals_conceded_1_ewm', 'goals_conceded_3_ewm',
    'goals_conceded_5_ewm', 'goals_scored_1_ewm', 'goals_scored_3_ewm', 'goals_scored_5_ewm', 'shots_1_ewm', 'shots_3_ewm', 'shots_5_ewm', 'key_passes_1_ewm',
    'key_passes_3_ewm', 'key_passes_5_ewm', 'npg_1_ewm', 'npg_3_ewm', 'npg_5_ewm', 'npxG_1_ewm', 'npxG_3_ewm', 'npxG_5_ewm', 'goals_1_ewm', 'goals_3_ewm',
    'goals_5_ewm', 'xG_1_ewm', 'xG_3_ewm', 'xG_5_ewm', 'xA_1_ewm', 'xA_3_ewm', 'xA_5_ewm', 'saves_1_ewm', 'saves_3_ewm', 'saves_5_ewm', 'starts_1_ewm', 'starts_3_ewm',
    'starts_5_ewm', 'yellow_cards_1_ewm', 'yellow_cards_3_ewm', 'yellow_cards_5_ewm', 'red_cards_1_ewm', 'red_cards_3_ewm', 'red_cards_5_ewm', 'assists_1_ewm',
    'assists_3_ewm', 'assists_5_ewm', 'clean_sheets_1_ewm', 'clean_sheets_3_ewm', 'clean_sheets_5_ewm', 'value_1_ewm', 'value_3_ewm', 'value_5_ewm', 'ict_index_1_ewm',
    'ict_index_3_ewm', 'ict_index_5_ewm', 'selected_1_ewm', 'selected_3_ewm', 'selected_5_ewm', 'transfers_in_1_ewm', 'transfers_in_3_ewm', 'transfers_in_5_ewm',
    'transfers_out_1_ewm', 'transfers_out_3_ewm', 'transfers_out_5_ewm', 'ownership_change_1_ewm', 'ownership_change_3_ewm', 'ownership_change_5_ewm',
    'xGChain_1_ewm', 'xGChain_3_ewm', 'xGChain_5_ewm',
    'xGBuildup_1_ewm', 'xGBuildup_3_ewm', 'xGBuildup_5_ewm', 'expected_goal_involvements_1_ewm', 'expected_goal_involvements_3_ewm', 'expected_goal_involvements_5_ewm',
    'selected_by_percent_1_ewm', 'selected_by_percent_3_ewm', 'selected_by_percent_5_ewm'
    ]

rolled_data_23_24 = data_to_roll.copy()

rolled_data_23_24.rename({'web_name': 'fpl_name'}, axis=1, inplace=True)

hist_data = rolled_data_23_24[
    (rolled_data_23_24['round'] > 5)  &
    (rolled_data_23_24['minutes_rolling_3'] > 15) &
    (rolled_data_23_24['minutes'] > 15)
    ]

hist_data

,fpl_id,round,understat_id,player_team,FPL_Name,Understat_Name,position,team,opponent_team,fpl_name,...,pts_bps_5_ewm,selected_by_percent_1_ewm,selected_by_percent_3_ewm,selected_by_percent_5_ewm,ownership_change_1_ewm,ownership_change_3_ewm,ownership_change_5_ewm,percentage_net_transfers_1_ewm,percentage_net_transfers_3_ewm,percentage_net_transfers_5_ewm
81,4,6,21952.0,Arsenal,Fábio Ferreira Vieira,Fábio Vieira,MID,Arsenal,Tottenham Hotspur,Fábio Ferreira Vieira,...,2.925926,0.001172,0.000964,0.000760,1522.0,2433.875000,2339.703704,0.000113,0.000321,0.000474
119,5,6,21952.0,Arsenal,Gabriel dos Santos Magalhães,Gabriel,DEF,Arsenal,Tottenham Hotspur,Gabriel dos Santos Magalhães,...,2.407407,0.122962,0.132046,0.158020,-102552.0,-30877.750000,-133014.074074,-0.012665,-0.045742,-0.104422
120,5,7,21959.0,Arsenal,Gabriel dos Santos Magalhães,Gabriel,DEF,Arsenal,Bournemouth,Gabriel dos Santos Magalhães,...,1.938272,0.114109,0.123077,0.143383,-56891.0,-43884.375000,-107639.716049,-0.008853,-0.027298,-0.072566
121,5,8,21965.0,Arsenal,Gabriel dos Santos Magalhães,Gabriel,DEF,Arsenal,Manchester City,Gabriel dos Santos Magalhães,...,3.292181,0.111717,0.117397,0.132828,-12208.0,-28046.187500,-75829.144033,-0.002391,-0.014845,-0.049174
122,5,9,21978.0,Arsenal,Gabriel dos Santos Magalhães,Gabriel,DEF,Arsenal,Chelsea,Gabriel dos Santos Magalhães,...,4.194787,0.106136,0.111766,0.123930,-55409.0,-41727.593750,-69022.429355,-0.005582,-0.010213,-0.034644
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24658,823,35,22240.0,Sheffield United,Oliver Arblaster,Oliver Arblaster,MID,Sheffield United,Newcastle United,Oliver Arblaster,...,1.222070,0.000062,0.000054,0.000046,91.0,110.812500,97.945283,0.000009,0.000008,0.000008
24659,823,36,22254.0,Sheffield United,Oliver Arblaster,Oliver Arblaster,MID,Sheffield United,Nottingham Forest,Oliver Arblaster,...,1.481380,0.000066,0.000060,0.000052,-58.0,26.406250,45.963522,0.000004,0.000006,0.000007
24660,823,37,22257.0,Sheffield United,Oliver Arblaster,Oliver Arblaster,MID,Sheffield United,Everton,Oliver Arblaster,...,1.654253,0.000069,0.000064,0.000058,30.0,28.203125,40.642348,0.000002,0.000004,0.000005
24661,823,38,22274.0,Sheffield United,Oliver Arblaster,Oliver Arblaster,MID,Sheffield United,Tottenham Hotspur,Oliver Arblaster,...,1.436169,0.000076,0.000070,0.000064,100.0,64.101562,60.428232,0.000008,0.000006,0.000006


# Model


### IRL for HEPs


In [51]:
def greedy_expert_strategy(players, hist_xp=None, K=15, max_team_ratio=3, corr_threshold=0.8):
    """
    Greedy selection that picks top xP_pred players while respecting
    team (club) caps and avoiding players highly correlated in historical xP.
    Returns a binary selection vector for the N players.
    (Paper: Heuristic-Guided Inverse RL -> Expert Trajectory Generation)
    """
    N = len(players)
    # Prioritize higher predicted points
    order = players.sort_values('xP_pred', ascending=False).index.tolist()
    selected = []
    team_counts = {}

    # Compute correlation matrix if hist_xp provided
    corr = None
    if hist_xp is not None:
        try:
            # Use Pearson correlation (linear relationship)
            corr = np.corrcoef(hist_xp, rowvar=False)
            corr = np.nan_to_num(corr)
        except Exception:
            corr = None

    for idx in order:
        if len(selected) >= K:
            break

        # Check FPL Club Constraint (Heuristic Rule 1)
        team = players.loc[idx, 'team']
        if team_counts.get(team, 0) >= max_team_ratio:
            continue

        # Check Correlation Constraint (Heuristic Rule 2: Correlation Control)
        if corr is not None and len(selected) > 0:
            # Calculate average positive correlation with already selected players
            selected_indices = [players.index.get_loc(s) for s in selected]
            player_idx = players.index.get_loc(idx)

            # Avg positive correlation with selected players
            correlations = [corr[player_idx, s_idx] for s_idx in selected_indices if corr[player_idx, s_idx] > 0]
            if correlations and np.mean(correlations) >= corr_threshold:
                continue

        selected.append(players.loc[idx, 'player_id'])
        team_counts[team] = team_counts.get(team, 0) + 1

    # Return selection vector based on original index (player_id)
    sel_vec = np.zeros(N, dtype=int)
    for p_id in selected:
        sel_vec[p_id] = 1
    return sel_vec

class RewardNetwork(nn.Module):
    """
    Simple reward network that combines features to produce scalar reward.
    (Paper: Multi-Objective Reward Learning)
    """
    def __init__(self, in_dim):
        super().__init__()
        # Simulates the combination of base features (return) and induced features (correlation/diversity)
        self.base = nn.Sequential(nn.Linear(in_dim, 64), nn.ReLU(), nn.Linear(64,32), nn.ReLU())
        self.ind_layer = nn.Sequential(nn.Linear(in_dim,32), nn.ReLU())
        self.out = nn.Linear(32+32, 1)

    def forward(self, x):
        b = self.base(x)
        i = self.ind_layer(x)
        concat = torch.cat([b,i], dim=-1)
        r = self.out(concat).squeeze(-1)
        return r

# -------------------------------------------------------------------------
# 3. REWARD LEARNING (MAXENT IRL)
# -------------------------------------------------------------------------

class RewardNetwork(nn.Module):
    """
    Simple reward network that combines features to produce scalar reward.
    (Paper: Multi-Objective Reward Learning)
    """
    def __init__(self, in_dim):
        super().__init__()
        # Simulates the combination of base features (return) and induced features (correlation/diversity)
        self.base = nn.Sequential(nn.Linear(in_dim, 64), nn.ReLU(), nn.Linear(64,32), nn.ReLU())
        self.ind_layer = nn.Sequential(nn.Linear(in_dim,32), nn.ReLU())
        self.out = nn.Linear(32+32, 1)

    def forward(self, x):
        b = self.base(x)
        i = self.ind_layer(x)
        concat = torch.cat([b,i], dim=-1)
        r = self.out(concat).squeeze(-1)
        return r

def maxent_irl_train(players, expert_sel_vec, epochs=200, lr=5e-3, device='cpu'):
    """
        Fits RewardNetwork to assign higher reward to expert-selected players.
        (Paper: Maximum Entropy Inverse Reinforcement Learning)
    """
    # Features: price, xP_pred, one-hot position (Simulating state representation)
    pos_map = {'GK':0,'DEF':1,'MID':2,'FWD':3}
    pos_onehot = np.zeros((len(players),4))
    for i,p in players.iterrows():
        pos_onehot[i, pos_map[p['position']]] = 1

    X = np.concatenate([players[['price','xP_pred']].values, pos_onehot], axis=1)
    X = torch.tensor(X, dtype=torch.float32, device=device)

    net = RewardNetwork(in_dim=X.shape[1]).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=lr)

    expert_mask = torch.tensor(expert_sel_vec, dtype=torch.float32, device=device)

    for ep in range(epochs):
        opt.zero_grad()
        r = net(X)  # Learned reward per player

        # MaxEntIRL loss (simplified): Encourages high reward for expert moves
        # while keeping the reward landscape smooth (entropy maximization).
        # Maximizing E_expert[R] equivalent to Minimizing -E_expert[R]

        # Safe division
        expert_sum = expert_mask.sum()
        if expert_sum == 0:
            expert_term = 0
        else:
            expert_term = (r * expert_mask).sum() / expert_sum

        loss = - expert_term
        loss += torch.logsumexp(r, dim=0) * 0.01             # Regularization (E_agent[e^R])

        loss.backward()
        opt.step()

        if (ep+1) % 50 == 0:
            print(f"IRL epoch {ep+1}/{epochs} loss={loss.item():.4f}")

    with torch.no_grad():
        learned_r = net(X).cpu().numpy()
        # Normalize learned reward to be positive and add to player table
        ptp = np.ptp(learned_r)
        if ptp == 0: ptp = 1e-8
        learned_r = (learned_r - learned_r.min()) / ptp
        players['learned_r'] = learned_r

    return net, players

# ---------------------- Simplified MaxEnt IRL training -------------------

def maxent_irl_train(players, expert_sel_vec, epochs=200, lr=5e-3, device='cpu'):
    """
    Fits RewardNetwork to assign higher reward to expert-selected players.
    (Paper: Maximum Entropy Inverse Reinforcement Learning)
    """
    # Features: price, xP_pred, one-hot position (Simulating state representation)
    pos_map = {'GK':0,'DEF':1,'MID':2,'FWD':3}
    pos_onehot = np.zeros((len(players),4))
    for i,p in players.iterrows():
        pos_onehot[i, pos_map[p['position']]] = 1

    X = np.concatenate([players[['price','xP_pred']].values, pos_onehot], axis=1)
    X = torch.tensor(X, dtype=torch.float32, device=device)

    net = RewardNetwork(in_dim=X.shape[1]).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=lr)

    expert_mask = torch.tensor(expert_sel_vec, dtype=torch.float32, device=device)

    for ep in range(epochs):
        opt.zero_grad()
        r = net(X)  # Learned reward per player

        # MaxEntIRL loss (simplified): Encourages high reward for expert moves
        # while keeping the reward landscape smooth (entropy maximization).
        loss = - (r * expert_mask).sum() / expert_mask.sum()  # Maximize E_expert[R]
        loss += torch.logsumexp(r, dim=0) * 0.01             # Regularization (E_agent[e^R])

        loss.backward()
        opt.step()
        if (ep+1) % 50 == 0:
            print(f"IRL epoch {ep+1}/{epochs} loss={loss.item():.4f}")

    with torch.no_grad():
        learned_r = net(X).cpu().numpy()
        # Normalize learned reward to be positive and add to player table
        # learned_r.ptp()
        learned_r = (learned_r - learned_r.min()) / (np.ptp(learned_r) + 1e-8)
        players['learned_r'] = learned_r

    return net, players

# ---------------------- Heterogeneous Graph Attention -------------------

class HGATPolicy(nn.Module):
    """
    A small HGAT that uses GATConv (if available) to produce player embeddings.
    (Paper: Heterogeneous Graph Policy Learning)
    """
    def __init__(self, in_dim, out_dim=32, heads=4):
        super().__init__()
        if GATConv is None:
            # fallback: MLP
            self.fallback = True
            self.mlp = nn.Sequential(nn.Linear(in_dim, 64), nn.ReLU(), nn.Linear(64, out_dim))
        else:
            self.fallback = False
            self.conv1 = GATConv(in_dim, 32, heads=heads, concat=True)
            self.conv2 = GATConv(32*heads, out_dim, heads=1, concat=False)

    def forward(self, x, edge_index=None):
        if self.fallback:
            return self.mlp(x)
        else:
            # Here, the HGAT would combine information from correlation graphs and team graphs.
            h = F.elu(self.conv1(x, edge_index))
            h = self.conv2(h, edge_index)
            return h

def ilp_select_squad(players, combined_score, budget=100.0, max_budget_per_pos=None):
    """
    Use Gurobi to select the optimal 11-man starting team and the captain.

    (Paper: Final Portfolio Optimization via Policy Output)
    """
    N = len(players)
    m = gp.Model("FPL_11_Man_Squad")
    m.Params.OutputFlag = 0 # Suppress Gurobi output

    # Decision Variables
    # x represents the 11 selected players (now the only selection variable)
    x = m.addVars(N, vtype=GRB.BINARY, name="x_selected")
    # z represents the Captain selection (1 player)
    z = m.addVars(N, vtype=GRB.BINARY, name="z_captain")

    # ------------------ OBJECTIVE FUNCTION ------------------
    # Maximize total score: Sum of (Selected Player Score) + (Captain Score Bonus)
    total_score = gp.quicksum(combined_score[i] * x[i] for i in range(N))
    captain_bonus = gp.quicksum(combined_score[i] * z[i] for i in range(N))
    m.setObjective(total_score + captain_bonus, GRB.MAXIMIZE)

    # ------------------ CONSTRAINTS ------------------

    # C1: Total size must be exactly 11 players
    m.addConstr(gp.quicksum(x[i] for i in range(N)) == 11, "Squad_Size_11")

    # C2: Budget constraint for the 11-man squad
    # Note: The budget is now used entirely for the 11 players.
    m.addConstr(gp.quicksum(players.loc[i,'price'] * x[i] for i in range(N)) <= budget, "Budget_Constraint")

    # C3: Club max 3 players in the 11-man squad
    clubs = players['team'].unique()
    for team in clubs:
        m.addConstr(gp.quicksum(x[i] for i in range(N) if players.loc[i,'team']==team) <= 3, f"Club_Limit_{team}")

    # C4: Captain selection (exactly 1 player)
    m.addConstr(gp.quicksum(z[i] for i in range(N)) == 1, "Captain_Select_1")

    # C5: Link between Captain and Selection: The captain must be one of the 11 selected players
    m.addConstrs((z[i] <= x[i] for i in range(N)), name="Captain_is_Selected")

    # C6: Formation constraint (Position limits for the 11 players)
    # Min/Max rules for the starting 11: 1 GK, 3-5 DEF, 2-5 MID, 1-3 FWD
    formation_limits = {
        "GK": (1, 1), # Exactly 1 GK must start
        "DEF": (3, 5),
        "MID": (2, 5),
        "FWD": (1, 3)
    }
    for pos, (min_req, max_req) in formation_limits.items():
        pos_selected = gp.quicksum(x[i] for i in range(N) if players.loc[i,'position']==pos)
        m.addConstr(pos_selected >= min_req, f"Selection_Min_{pos}")
        m.addConstr(pos_selected <= max_req, f"Selection_Max_{pos}")


    m.optimize()

    # Extract Results
    selected_ids = []
    captain_id = -1

    if m.status == GRB.OPTIMAL:
        for i in range(N):
            if x[i].X > 0.5:
                selected_ids.append(players.loc[i, 'player_id'])
            if z[i].X > 0.5:
                captain_id = players.loc[i, 'player_id']

    # Since we only selected 11, the "starter" list is the same as the "squad" list
    return selected_ids, selected_ids, captain_id

# ----------------------------- Putting it together -----------------------

def run_pipeline(players):
    """Main function to run the IRL-HGAT-ILP FPL optimization pipeline."""
    # Step 1: Data Generation
    # players, hist = make_dummy_players(n_players=90, seed=42)

    # players
    hist = None

    # Step 2: Generate Expert Strategy (Heuristic-Guided Trajectory)
    # Note: K=11 for the expert selection now, to match the final goal.
    expert_sel = greedy_expert_strategy(
        players,
        hist_xp=hist,
        K=11, # Changed to 11
        max_team_ratio=3,
        corr_threshold=0.85 # Tighter correlation constraint for the expert
    )
    print(f"Expert selected count (11-man heuristic): {expert_sel.sum()}")

    # Step 3: Train Simplified MaxEnt IRL (Multi-Objective Reward Learning)
    net, players = maxent_irl_train(players.copy(), expert_sel, epochs=200, lr=5e-3)

    # Step 4: HGAT Policy (Feature Embedding, simulates cross-player understanding)
    N = len(players)
    # Build graph edges based on club affiliation (Industry/Sector Graph in the paper)

    edge_idx = []
    for i in range(N):
        for j in range(i + 1, N):
            # print(i, j)
            if players.loc[i,'team'] == players.loc[j,'team']:
                # Bidirectional edge
                edge_idx.append([i, j])
                edge_idx.append([j, i])
    edge_index = torch.tensor(edge_idx, dtype=torch.long).t().contiguous() if edge_idx else None

    # Node features: price, xP_pred, learned_r, one-hot pos
    pos_map = {'GK': 0, 'DEF': 1, 'MID': 2, 'FWD': 3}
    pos_onehot = np.zeros((N, 4))
    for i, p in players.iterrows():
        pos_onehot[i, pos_map[p['position']]] = 1

    X = np.concatenate([
        players[['price', 'xP_pred', 'learned_r']].values,
        pos_onehot
    ], axis=1)
    X_t = torch.tensor(X, dtype=torch.float32)

    policy = HGATPolicy(in_dim=X.shape[1])
    with torch.no_grad():
        emb = policy(X_t, edge_index)
    print(f"HGAT Policy Embeddings Shape: {emb.shape}")

    # Step 5: Final ILP Optimization using Combined Score
    # We combine the "raw return" (xP_pred) with the "learned reward" (risk/diversity value)
    combined_score = (
        0.6 * players['xP_pred'].values +
        0.4 * players['learned_r'].values
    )

    # Note: The 'squad_sel_ids' and 'starter_sel_ids' are now identical (the 11 players)
    selected_ids, _, captain_id = ilp_select_squad(
        players,
        combined_score,
        budget=100.0 # Adjusted budget for a likely 11-man team
    )

    print("\n--- OPTIMAL FPL TEAM (IRL-HGAT-ILP RESULT) ---")
    if selected_ids and len(selected_ids) == 11:
        squad_df = players[players['player_id'].isin(selected_ids)].copy()
        squad_df['Is_Captain'] = squad_df['player_id'] == captain_id

        # Calculate total score from the optimal selection
        total_xp_maximized = squad_df['xP_pred'].sum()
        captain_xp = squad_df[squad_df['Is_Captain']]['xP_pred'].iloc[0]
        total_xp_maximized += captain_xp # Add bonus points

        print(squad_df[['name','team','position','price','xP_pred', 'Is_Captain']].sort_values(by=['position', 'price'], ascending=[False, False]))
        print("\n--- SUMMARY ---")
        print(f"Total Team Cost: £{squad_df['price'].sum():.1f}m")
        print(f"Maximized Raw xP (11 Players + Captain Bonus): {total_xp_maximized:.2f} points")

        # Display Formation:
        formation = squad_df.groupby('position').size().to_dict()
        print(f"Formation Selected: {formation.get('DEF', 0)}-{formation.get('MID', 0)}-{formation.get('FWD', 0)}")
    else:
        print("No feasible solution found (or incorrect number of players selected).")

    return squad_df


In [15]:
common_features = [
    'expected_goals_conceded_rolling_5', 'influence_5_ewm', 'lose_prob', 'transfers_in_5_ewm', 'xA_5_ewm', 'percentage_net_transfers_5_ewm',
    'xGBuildup_3_ewm', 'xGBuildup_5_ewm', 'ict_index_rolling_5', 'assists_3_ewm', 'ownership_change_rolling_3',
    'value', 'xP_5_ewm', 'selected_by_percent_5_ewm', 'selected_by_percent_rolling_1', 'win_prob', 'influence_rolling_5',
    'selected_by_percent', 'npg_3_ewm',  'minutes_rolling_5',
    'selected_by_percent_rolling_5', 'npxG_rolling_3', 'expected_goals_conceded_5_ewm', 'xGBuildup_rolling_3',
    'transfers_in_rolling_1', 'value_rolling_5', 'expected_goal_involvements_rolling_1', 'expected_goals_conceded_rolling_1', 'xGChain_rolling_1',
    'ownership_change', 'influence_3_ewm', 'transfers_out', 'xP_rolling_1', 'xP_3_ewm', 'selected_rolling_5',
    'transfers_out_rolling_1', 'expected_goals_conceded_3_ewm', 'percentage_net_transfers_3_ewm', 'xGChain_rolling_3', 'xGBuildup_rolling_5',
    'xGChain_rolling_5', 'goals_conceded_5_ewm', 'xP_rolling_3', 'percentage_net_transfers', 'yellow_cards_5_ewm', 'npxG_rolling_5',
    'transfers_out_rolling_5', 'xGChain_5_ewm', 'yellow_cards_3_ewm', 'creativity_rolling_1', 'ict_index_rolling_3',  'transfers_in_rolling_3',
    'pts_bps_3_ewm', 'influence_rolling_1', 'xA_rolling_5', 'ownership_change_rolling_1', 'transfers_in', 'clean_sheets_3_ewm', 'selected',
    'expected_goals_conceded_rolling_3', 'pts_bps_rolling_3', 'draw_prob', 'xA_rolling_3',
    'xGBuildup_rolling_1', 'ownership_change_3_ewm', 'percentage_net_transfers_rolling_5', 'ict_index_rolling_1',
    'ownership_change_rolling_5', 'clean_sheets_5_ewm', 'creativity_rolling_3', 'percentage_net_transfers_rolling_3', 'transfers_out_5_ewm',
    'transfers_in_3_ewm', 'creativity_rolling_5', 'transfers_in_rolling_5', 'goals_conceded_3_ewm', 'pts_bps_rolling_1', 'percentage_net_transfers_rolling_1',
    'influence_rolling_3', 'pts_bps_5_ewm',  'pts_bps_rolling_5', 'team_a_difficulty',  'team_h_difficulty'
    # 'opp_elo', 'elo', 'opp_clean_sheets',
    ]
rolled_data_23_24[common_features]

,expected_goals_conceded_rolling_5,influence_5_ewm,lose_prob,transfers_in_5_ewm,xA_5_ewm,percentage_net_transfers_5_ewm,xGBuildup_3_ewm,xGBuildup_5_ewm,ict_index_rolling_5,assists_3_ewm,...,creativity_rolling_5,transfers_in_rolling_5,goals_conceded_3_ewm,pts_bps_rolling_1,percentage_net_transfers_rolling_1,influence_rolling_3,pts_bps_5_ewm,pts_bps_rolling_5,team_a_difficulty,team_h_difficulty
23,0.000,0.000000,0.622,0.000000,0.0,0.000000e+00,0.000000,0.000000,0.00,0.000000,...,0.00,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,3.0,5.0
24,0.026,0.133333,0.739,90.000000,0.0,1.636684e-03,0.012268,0.008179,0.02,0.000000,...,0.14,54.0,0.0,1.0,0.004910,0.133333,0.333333,0.2,2.0,5.0
26,0.056,0.192593,0.808,194.666667,0.0,-8.646276e-04,0.046122,0.041906,0.06,0.000000,...,0.40,193.2,0.0,0.0,-0.004508,0.333333,0.370370,0.4,2.0,5.0
47,0.000,0.000000,0.050,0.000000,0.0,0.000000e+00,0.000000,0.000000,0.00,0.000000,...,0.00,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,5.0,2.0
61,0.000,0.003083,0.622,1.378759,0.0,-6.906466e-07,0.000000,0.000000,0.00,0.000061,...,0.00,0.0,0.0,0.0,0.000000,0.000000,0.006851,0.0,3.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25268,0.000,0.000000,0.696,0.000000,0.0,0.000000e+00,0.000000,0.000000,0.00,0.000000,...,0.00,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,4.0,5.0
25269,0.010,0.333333,0.708,18.000000,0.0,2.910150e-06,0.000000,0.000000,0.02,0.000000,...,0.02,10.8,0.0,1.0,0.000009,0.333333,0.333333,0.2,2.0,3.0
25306,0.000,0.000000,0.199,0.000000,0.0,0.000000e+00,0.000000,0.000000,0.00,0.000000,...,0.00,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,4.0,2.0
25307,0.000,0.000000,0.431,0.000000,0.0,0.000000e+00,0.738057,0.492038,0.00,0.000000,...,0.00,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,2.0,3.0


In [16]:
# rolled_data_23_24[cols]
rolled_data_23_24.columns.tolist()

features = [
    'xP', 'selected_by_percent', 'ownership_change', 'assists_rolling_1', 'assists_rolling_3', 'assists_rolling_5', 'clean_sheets_rolling_1', 'clean_sheets_rolling_3',
    'clean_sheets_rolling_5', 'creativity_rolling_1', 'creativity_rolling_3', 'creativity_rolling_5', 'element_rolling_1', 'element_rolling_3', 'element_rolling_5',
    'expected_assists_rolling_1', 'expected_assists_rolling_3', 'expected_assists_rolling_5', 'expected_goal_involvements_rolling_1',
    'expected_goal_involvements_rolling_3', 'expected_goal_involvements_rolling_5', 'expected_goals_rolling_1', 'expected_goals_rolling_3', 'expected_goals_rolling_5',
    'expected_goals_conceded_rolling_1', 'expected_goals_conceded_rolling_3', 'expected_goals_conceded_rolling_5', 'goals_conceded_rolling_1', 'goals_conceded_rolling_3',
    'goals_conceded_rolling_5', 'goals_scored_rolling_1', 'goals_scored_rolling_3', 'goals_scored_rolling_5', 'ict_index_rolling_1', 'ict_index_rolling_3',
    'ict_index_rolling_5', 'influence_rolling_1', 'influence_rolling_3', 'influence_rolling_5', 'minutes_rolling_1', 'minutes_rolling_3', 'minutes_rolling_5',
    'own_goals_rolling_1', 'own_goals_rolling_3', 'own_goals_rolling_5', 'penalties_missed_rolling_1', 'penalties_missed_rolling_3', 'penalties_missed_rolling_5',
    'penalties_saved_rolling_1', 'penalties_saved_rolling_3', 'penalties_saved_rolling_5', 'red_cards_rolling_1', 'red_cards_rolling_3', 'red_cards_rolling_5',
    'saves_rolling_1', 'saves_rolling_3', 'saves_rolling_5', 'selected_rolling_1', 'selected_rolling_3', 'selected_rolling_5', 'starts_rolling_1', 'starts_rolling_3',
    'starts_rolling_5', 'team_a_score_rolling_1', 'team_a_score_rolling_3', 'team_a_score_rolling_5', 'team_h_score_rolling_1', 'team_h_score_rolling_3',
    'team_h_score_rolling_5', 'threat_rolling_1', 'threat_rolling_3', 'threat_rolling_5', 'transfers_balance_rolling_1', 'transfers_balance_rolling_3',
    'transfers_balance_rolling_5', 'transfers_in_rolling_1', 'transfers_in_rolling_3', 'transfers_in_rolling_5', 'transfers_out_rolling_1', 'transfers_out_rolling_3',
    'transfers_out_rolling_5', 'value_rolling_1', 'value_rolling_3', 'value_rolling_5', 'yellow_cards_rolling_1', 'yellow_cards_rolling_3', 'yellow_cards_rolling_5',
    'goals_rolling_1', 'goals_rolling_3', 'goals_rolling_5', 'shots_rolling_1', 'shots_rolling_3', 'shots_rolling_5', 'xG_rolling_1', 'xG_rolling_3', 'xG_rolling_5',
    'h_goals_rolling_1', 'h_goals_rolling_3', 'h_goals_rolling_5', 'a_goals_rolling_1', 'a_goals_rolling_3', 'a_goals_rolling_5', 'xA_rolling_1', 'xA_rolling_3',
    'xA_rolling_5', 'key_passes_rolling_1', 'key_passes_rolling_3', 'key_passes_rolling_5', 'npg_rolling_1', 'npg_rolling_3', 'npg_rolling_5', 'npxG_rolling_1',
    'npxG_rolling_3', 'npxG_rolling_5', 'xGChain_rolling_1', 'xGChain_rolling_3', 'xGChain_rolling_5', 'xGBuildup_rolling_1', 'xGBuildup_rolling_3',
    'xGBuildup_rolling_5', 'xP_rolling_1', 'xP_rolling_3', 'xP_rolling_5', 'pts_bps_rolling_1', 'pts_bps_rolling_3', 'pts_bps_rolling_5', 'selected_by_percent_rolling_1',
    'selected_by_percent_rolling_3', 'selected_by_percent_rolling_5', 'ownership_change_rolling_1', 'ownership_change_rolling_3', 'ownership_change_rolling_5',
    'assists_1_ewm', 'assists_3_ewm', 'assists_5_ewm', 'clean_sheets_1_ewm', 'clean_sheets_3_ewm', 'clean_sheets_5_ewm', 'creativity_1_ewm', 'creativity_3_ewm',
    'creativity_5_ewm', 'element_1_ewm', 'element_3_ewm', 'element_5_ewm', 'expected_assists_1_ewm', 'expected_assists_3_ewm', 'expected_assists_5_ewm',
    'expected_goal_involvements_1_ewm', 'expected_goal_involvements_3_ewm', 'expected_goal_involvements_5_ewm', 'expected_goals_1_ewm', 'expected_goals_3_ewm',
    'expected_goals_5_ewm', 'expected_goals_conceded_1_ewm', 'expected_goals_conceded_3_ewm', 'expected_goals_conceded_5_ewm', 'goals_conceded_1_ewm',
    'goals_conceded_3_ewm', 'goals_conceded_5_ewm', 'goals_scored_1_ewm', 'goals_scored_3_ewm', 'goals_scored_5_ewm', 'ict_index_1_ewm', 'ict_index_3_ewm',
    'ict_index_5_ewm', 'influence_1_ewm', 'influence_3_ewm', 'influence_5_ewm', 'minutes_1_ewm', 'minutes_3_ewm', 'minutes_5_ewm', 'own_goals_1_ewm', 'own_goals_3_ewm',
    'own_goals_5_ewm', 'penalties_missed_1_ewm', 'penalties_missed_3_ewm', 'penalties_missed_5_ewm', 'penalties_saved_1_ewm', 'penalties_saved_3_ewm',
    'penalties_saved_5_ewm', 'red_cards_1_ewm', 'red_cards_3_ewm', 'red_cards_5_ewm', 'saves_1_ewm', 'saves_3_ewm', 'saves_5_ewm', 'selected_1_ewm', 'selected_3_ewm',
    'selected_5_ewm', 'starts_1_ewm', 'starts_3_ewm', 'starts_5_ewm', 'team_a_score_1_ewm', 'team_a_score_3_ewm', 'team_a_score_5_ewm', 'team_h_score_1_ewm',
    'team_h_score_3_ewm', 'team_h_score_5_ewm', 'threat_1_ewm', 'threat_3_ewm', 'threat_5_ewm', 'transfers_balance_1_ewm', 'transfers_balance_3_ewm',
    'transfers_balance_5_ewm', 'transfers_in_1_ewm', 'transfers_in_3_ewm', 'transfers_in_5_ewm', 'transfers_out_1_ewm', 'transfers_out_3_ewm', 'transfers_out_5_ewm',
    'value_1_ewm', 'value_3_ewm', 'value_5_ewm', 'yellow_cards_1_ewm', 'yellow_cards_3_ewm', 'yellow_cards_5_ewm', 'goals_1_ewm', 'goals_3_ewm', 'goals_5_ewm',
    'shots_1_ewm', 'shots_3_ewm', 'shots_5_ewm', 'xG_1_ewm', 'xG_3_ewm', 'xG_5_ewm', 'h_goals_1_ewm', 'h_goals_3_ewm', 'h_goals_5_ewm', 'a_goals_1_ewm', 'a_goals_3_ewm',
    'a_goals_5_ewm', 'xA_1_ewm', 'xA_3_ewm', 'xA_5_ewm', 'key_passes_1_ewm', 'key_passes_3_ewm', 'key_passes_5_ewm', 'npg_1_ewm', 'npg_3_ewm', 'npg_5_ewm', 'npxG_1_ewm',
    'npxG_3_ewm', 'npxG_5_ewm', 'xGChain_1_ewm', 'xGChain_3_ewm', 'xGChain_5_ewm', 'xGBuildup_1_ewm', 'xGBuildup_3_ewm', 'xGBuildup_5_ewm', 'xP_1_ewm', 'xP_3_ewm',
    'xP_5_ewm', 'pts_bps_1_ewm', 'pts_bps_3_ewm', 'pts_bps_5_ewm', 'selected_by_percent_1_ewm', 'selected_by_percent_3_ewm', 'selected_by_percent_5_ewm',
    'ownership_change_1_ewm', 'ownership_change_3_ewm', 'ownership_change_5_ewm'
]

## Train


In [ ]:

working_data = rolled_data_23_24[rolled_data_23_24['round'] < 31].copy()

# Prepare your features (X) and target (y)
# Let's use fpl_id and round as basic features
X = working_data[common_features]
y = working_data['xP']

# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create LightGBM datasets
train_data = lgbm.Dataset(X_train, label=y_train)
test_data = lgbm.Dataset(X_test, label=y_test, reference=train_data)

# Set parameters for LightGBM
params = {
    'objective': 'regression',
    'metric': 'mse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1
}

# Train the model
model = lgbm.train(params, train_data, num_boost_round=1000)

# Predict on the test set
y_pred = model.predict(X_test, num_iteration=model.best_iteration)
# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error: {mse}')



Mean Squared Error: 1.3182023234002525


## Predict


In [40]:
rolled_data_23_24['fpl_name']

23       Cédric Alves Soares
24       Cédric Alves Soares
26       Cédric Alves Soares
47            Mohamed Elneny
61            Mohamed Elneny
                ...         
25268            Mikey Moore
25269            Mikey Moore
25306            Tyrese Hall
25307            Tyrese Hall
25345           Lander Emery
Name: fpl_name, Length: 11802, dtype: object

In [66]:
results = []
for round in range(31, 39):
    predicition_data = rolled_data_23_24[rolled_data_23_24['round'] == round].copy()

    # Prepare your features (X) and target (y)
    # Let's use fpl_id and round as basic features
    X = predicition_data[common_features]
    y = predicition_data[['fpl_id', 'fpl_name', 'value', 'team', 'position', 'xP']].copy()

    y_pred = model.predict(X, num_iteration=model.best_iteration)

    y['xP_pred'] =  y_pred
    y['round']  =  round

    results.append(y)
preds = pd.concat(results)
# preds = preds.rename(columns={'fpl_name'})

In [67]:
preds

,fpl_id,fpl_name,value,team,position,xP,xP_pred,round
144,5,Gabriel dos Santos Magalhães,5.3,Arsenal,DEF,5.8,5.609931,31
182,6,Kai Havertz,7.2,Arsenal,MID,6.5,6.663123,31
258,9,Jorge Luiz Frello Filho,5.3,Arsenal,MID,3.5,2.462454,31
372,12,Gabriel Martinelli Silva,7.6,Arsenal,MID,2.2,3.514694,31
410,13,Eddie Nketiah,5.1,Arsenal,FWD,1.8,1.157538,31
...,...,...,...,...,...,...,...,...
25193,854,Finley Munroe,4.0,Aston Villa,DEF,0.7,0.226679,38
25231,856,Louis Jackson,4.0,Aston Villa,DEF,0.0,-0.170540,38
25269,860,Mikey Moore,4.5,Tottenham Hotspur,MID,1.3,0.253197,38
25307,862,Tyrese Hall,4.5,Newcastle United,MID,1.0,0.877478,38


### Optimize


#### GW31


##### MVT


##### Select 11


In [73]:
players_31 = preds[preds['round'] == 31].copy()


players_31['player_id'] = list(range(len(players_31)))

players_31.rename(columns={'fpl_name': 'name', 'value': 'price'}, inplace=True)
players_31 = players_31.reset_index(drop=True)

selected_31 = run_pipeline(players_31)

Expert selected count (11-man heuristic): 11
IRL epoch 50/200 loss=-176.5746
IRL epoch 100/200 loss=-2069.3298
IRL epoch 150/200 loss=-8972.5986
IRL epoch 200/200 loss=-24748.0410
HGAT Policy Embeddings Shape: torch.Size([306, 32])

--- OPTIMAL FPL TEAM (IRL-HGAT-ILP RESULT) ---
                     name               team position  price   xP_pred  \
134         Mohamed Salah          Liverpool      MID   13.3  7.303492   
207         Son Heung-min  Tottenham Hotspur      MID   10.1  8.231848   
131             Luis Díaz          Liverpool      MID    7.5  6.191402   
242          Moussa Diaby        Aston Villa      MID    6.3  6.225750   
154           Cole Palmer    Manchester City      MID    5.9  7.374796   
45      David Raya Martin            Arsenal       GK    5.0  5.675991   
177        Alexander Isak   Newcastle United      FWD    7.7  8.647824   
124  Darwin Núñez Ribeiro          Liverpool      FWD    7.6  5.827995   
7          William Saliba            Arsenal      DEF 

##### Get Scores


In [77]:
# Get total points
players_selected_31 = players_31[players_31['fpl_id'].isin(selected_31['fpl_id'])].copy()
# players_selected_31 = players_selected_31.copy()
# 1. Create the rank column based on 'xp_preds'
# method='dense' ensures ties get the same rank, and the next rank is sequential
players_selected_31['xp_rank'] = players_selected_31['xP_pred'].rank(method='dense', ascending=False).astype(int)

# 2. Define the conditions and their corresponding multipliers
conditions = [
    players_selected_31['xp_rank'] == 1,  # Highest xp_preds
    players_selected_31['xp_rank'] == 2   # Second highest xp_preds
]
multipliers = [2.0, 1.5] # [multiplier for rank 1, multiplier for rank 2]

# 3. Use np.select() to apply the multiplier
# The 'default=1.0' means any rank other than 1 or 2 gets a 1.0 multiplier
players_selected_31['score'] = players_selected_31['xP'] * np.select(conditions, multipliers, default=1.0)
total_points_31 = sum(players_selected_31['score'].values)
print(total_points_31, players_selected_31['xP'].values, '--------->', players_selected_31['xP_pred'].values)

players_selected_31[['name', 'team', 'position', 'xP_pred', 'xP', 'xp_rank', 'score']]

94.05000000000001 [ 6.8  9.2  4.   5.2  5.3  7.   4.  16.   9.7  8.3  4.7] ---------> [6.09604281 6.86094813 5.03692396 5.67599134 5.82799498 6.19140209
 7.3034916  7.37479614 8.64782385 8.23184785 6.22575045]


,name,team,position,xP_pred,xP,xp_rank,score
7,William Saliba,Arsenal,DEF,6.096043,6.8,8,6.80
12,Benjamin White,Arsenal,DEF,6.860948,9.2,5,9.20
21,Ezri Konsa Ngoyo,Aston Villa,DEF,5.036924,4.0,11,4.00
45,David Raya Martin,Arsenal,GK,5.675991,5.2,10,5.20
124,Darwin Núñez Ribeiro,Liverpool,FWD,5.827995,5.3,9,5.30
131,Luis Díaz,Liverpool,MID,6.191402,7.0,7,7.00
134,Mohamed Salah,Liverpool,MID,7.303492,4.0,4,4.00
154,Cole Palmer,Manchester City,MID,7.374796,16.0,3,16.00
177,Alexander Isak,Newcastle United,FWD,8.647824,9.7,1,19.40
207,Son Heung-min,Tottenham Hotspur,MID,8.231848,8.3,2,12.45


##### Rank


In [114]:
fg_managers_31 = fg_managers[fg_managers['gameweek'] == 31]
fg_managers_31[['points']]

position_31 = (fg_managers_31['points'] > total_points_31).sum() + 1

# 4. Display result
total_managers_31 = len(fg_managers_31)
percentile_31 = percentile_34 = 100 * (1 - (position_31 - 1) / total_managers_31)

print(total_points_31, total_managers_31)

print(f"Your predicted player’s total of {total_points_31} points would rank:")
print(f"→ Position: {position_31} out of {total_managers_31} managers")
print(f"→ Percentile: Top {percentile_31:.2f}%")


# Optional: show nearby ranks for context
fg_sorted_31 = fg_managers_31.sort_values('points', ascending=False).reset_index(drop=True)

nearby_31 = fg_sorted_31.iloc[max(position_31-15, 0):position_31+2][['points','prize']]

player_prize_31 = fg_sorted_31.iloc[position_31 - 1]['prize']
if pd.isna(player_prize_31):
    player_prize_31 = "No prize"

print(f"→ Prize at position {position_31}: {player_prize_31}")
print("\nClosest scores around you:")
print(nearby_31)

94.05000000000001 217
Your predicted player’s total of 94.05000000000001 points would rank:
→ Position: 1 out of 217 managers
→ Percentile: Top 100.00%
→ Prize at position 1: R 3,890.25

Closest scores around you:
   points       prize
0    93.0  R 3,890.25
1    92.5  R 1,695.75
2    89.0  R 1,097.25


#### GW32


##### Select 11


In [80]:
players_32 = preds[preds['round'] == 32].copy()


players_32['player_id'] = list(range(len(players_32)))

players_32.rename(columns={'fpl_name': 'name', 'value': 'price'}, inplace=True)
players_32 = players_32.reset_index(drop=True)

selected_32 = run_pipeline(players_32)

Expert selected count (11-man heuristic): 11


IRL epoch 50/200 loss=-410.7194
IRL epoch 100/200 loss=-4789.6138
IRL epoch 150/200 loss=-20798.1602
IRL epoch 200/200 loss=-57435.3398
HGAT Policy Embeddings Shape: torch.Size([328, 32])

--- OPTIMAL FPL TEAM (IRL-HGAT-ILP RESULT) ---
                   name                     team position  price   xP_pred  \
141       Mohamed Salah                Liverpool      MID   13.4  6.085524   
8       Martin Ødegaard                  Arsenal      MID    8.5  6.133420   
222        Jarrod Bowen          West Ham United      MID    7.9  6.493179   
166         Cole Palmer          Manchester City      MID    6.0  8.208829   
175  Alejandro Garnacho        Manchester United      MID    4.9  6.266259   
44         Mark Flekken                Brentford       GK    4.6  5.010144   
188      Alexander Isak         Newcastle United      FWD    7.9  8.492962   
200          Chris Wood        Nottingham Forest      FWD    4.8  5.560727   
14       Benjamin White                  Arsenal      DEF    5

##### Get Scores


In [81]:
# Get total points
players_selected_32 = players_32[players_32['fpl_id'].isin(selected_32['fpl_id'])].copy()
# players_selected_32 = players_selected_32.copy()
# 1. Create the rank column based on 'xp_preds'
# method='dense' ensures ties get the same rank, and the next rank is sequential
players_selected_32['xp_rank'] = players_selected_32['xP_pred'].rank(method='dense', ascending=False).astype(int)

# 2. Define the conditions and their corresponding multipliers
conditions = [
    players_selected_32['xp_rank'] == 1,  # Highest xp_preds
    players_selected_32['xp_rank'] == 2   # Second highest xp_preds
]
multipliers = [2.0, 1.5] # [multiplier for rank 1, multiplier for rank 2]

# 3. Use np.select() to apply the multiplier
# The 'default=1.0' means any rank other than 1 or 2 gets a 1.0 multiplier
players_selected_32['score'] = players_selected_32['xP'] * np.select(conditions, multipliers, default=1.0)
total_points_32 = sum(players_selected_32['score'].values)
print(total_points_32, players_selected_32['xP'].values, '--------->', players_selected_32['xP_pred'].values)

players_selected_32[['name', 'team', 'position', 'xP_pred', 'xP', 'xp_rank', 'score']]

86.24999999999999 [ 5.8  6.2  8.   3.3  4.7 13.7  6.   7.8  6.1  4.   6. ] ---------> [6.13342047 5.38585917 6.51445595 5.01014427 6.08552371 8.20882872
 6.26625891 8.49296191 5.56072743 6.49317888 6.05538801]


,name,team,position,xP_pred,xP,xp_rank,score
8,Martin Ødegaard,Arsenal,MID,6.133420,5.8,6,5.80
10,William Saliba,Arsenal,DEF,5.385859,6.2,10,6.20
14,Benjamin White,Arsenal,DEF,6.514456,8.0,3,8.00
44,Mark Flekken,Brentford,GK,5.010144,3.3,11,3.30
141,Mohamed Salah,Liverpool,MID,6.085524,4.7,7,4.70
166,Cole Palmer,Manchester City,MID,8.208829,13.7,2,20.55
175,Alejandro Garnacho,Manchester United,MID,6.266259,6.0,5,6.00
188,Alexander Isak,Newcastle United,FWD,8.492962,7.8,1,15.60
200,Chris Wood,Nottingham Forest,FWD,5.560727,6.1,9,6.10
222,Jarrod Bowen,West Ham United,MID,6.493179,4.0,4,4.00


##### Rank


In [116]:
fg_managers_32 = fg_managers[fg_managers['gameweek'] == 32]
fg_managers_32[['points']]

position_32 = (fg_managers_32['points'] > total_points_32).sum() + 1

# 4. Display result
total_managers_32 = len(fg_managers_32)
percentile_32 = percentile_34 = 100 * (1 - (position_32 - 1) / total_managers_32)

print(total_points_32, total_managers_32)

print(f"Your predicted player’s total of {total_points_32} points would rank:")
print(f"→ Position: {position_32} out of {total_managers_32} managers")
print(f"→ Percentile: Top {percentile_32:.2f}%")


# Optional: show nearby ranks for context
fg_sorted_32 = fg_managers_32.sort_values('points', ascending=False).reset_index(drop=True)

nearby_32 = fg_sorted_32.iloc[max(position_32-15, 0):position_32+2][['points','prize']]

player_prize_32 = fg_sorted_32.iloc[position_32 - 1]['prize']
if pd.isna(player_prize_32):
    player_prize_32 = "No prize"

print(f"→ Prize at position {position_32}: {player_prize_32}")
print("\nClosest scores around you:")
print(nearby_32)

86.24999999999999 234
Your predicted player’s total of 86.24999999999999 points would rank:
→ Position: 8 out of 234 managers
→ Percentile: Top 97.01%
→ Prize at position 8: R 99.75

Closest scores around you:
   points       prize
0   100.5  R 3,890.25
1    93.5  R 1,695.75
2    90.0  R 1,097.25
3    88.0    R 522.50
4    87.5    R 299.25
5    87.5    R 299.25
6    87.5    R 299.25
7    86.0     R 99.75
8    85.0     R 99.75
9    84.5     R 99.75


#### GW33


##### Select 11


In [83]:
players_33 = preds[preds['round'] == 33].copy()


players_33['player_id'] = list(range(len(players_33)))

players_33.rename(columns={'fpl_name': 'name', 'value': 'price'}, inplace=True)
players_33 = players_33.reset_index(drop=True)

selected_33 = run_pipeline(players_33)

Expert selected count (11-man heuristic): 11
IRL epoch 50/200 loss=-283.4347
IRL epoch 100/200 loss=-3258.2295
IRL epoch 150/200 loss=-14097.5879
IRL epoch 200/200 loss=-38869.8125
HGAT Policy Embeddings Shape: torch.Size([328, 32])

--- OPTIMAL FPL TEAM (IRL-HGAT-ILP RESULT) ---
                     name               team position  price   xP_pred  \
139         Mohamed Salah          Liverpool      MID   13.5  6.694251   
154       Kevin De Bruyne    Manchester City      MID   10.4  8.919511   
136             Luis Díaz          Liverpool      MID    7.7  7.563138   
1             Kai Havertz            Arsenal      MID    7.4  7.022847   
162           Cole Palmer    Manchester City      MID    6.1  8.002939   
47      David Raya Martin            Arsenal       GK    5.1  5.205077   
130  Darwin Núñez Ribeiro          Liverpool      FWD    7.7  7.926663   
194            Chris Wood  Nottingham Forest      FWD    4.9  6.003879   
14         Benjamin White            Arsenal      DEF

##### Get Scores


In [84]:
# Get total points
players_selected_33 = players_33[players_33['fpl_id'].isin(selected_33['fpl_id'])].copy()
# players_selected_33 = players_selected_33.copy()
# 1. Create the rank column based on 'xp_preds'
# method='dense' ensures ties get the same rank, and the next rank is sequential
players_selected_33['xp_rank'] = players_selected_33['xP_pred'].rank(method='dense', ascending=False).astype(int)

# 2. Define the conditions and their corresponding multipliers
conditions = [
    players_selected_33['xp_rank'] == 1,  # Highest xp_preds
    players_selected_33['xp_rank'] == 2   # Second highest xp_preds
]
multipliers = [2.0, 1.5] # [multiplier for rank 1, multiplier for rank 2]

# 3. Use np.select() to apply the multiplier
# The 'default=1.0' means any rank other than 1 or 2 gets a 1.0 multiplier
players_selected_33['score'] = players_selected_33['xP'] * np.select(conditions, multipliers, default=1.0)
total_points_33 = sum(players_selected_33['score'].values)
print(total_points_33, players_selected_33['xP'].values, '--------->', players_selected_33['xP_pred'].values)

players_selected_33[['name', 'team', 'position', 'xP_pred', 'xP', 'xp_rank', 'score']]

88.95 [ 6.3  6.   5.   4.3  6.7  5.   7.3  6.  16.5  6.   4.3] ---------> [7.02284738 5.77688486 5.2050766  7.92666327 7.56313789 6.69425099
 8.91951128 5.67338574 8.00293908 6.00387925 4.91344246]


,name,team,position,xP_pred,xP,xp_rank,score
1,Kai Havertz,Arsenal,MID,7.022847,6.3,5,6.30
14,Benjamin White,Arsenal,DEF,5.776885,6.0,8,6.00
47,David Raya Martin,Arsenal,GK,5.205077,5.0,10,5.00
130,Darwin Núñez Ribeiro,Liverpool,FWD,7.926663,4.3,3,4.30
136,Luis Díaz,Liverpool,MID,7.563138,6.7,4,6.70
139,Mohamed Salah,Liverpool,MID,6.694251,5.0,6,5.00
154,Kevin De Bruyne,Manchester City,MID,8.919511,7.3,1,14.60
160,Rico Lewis,Manchester City,DEF,5.673386,6.0,9,6.00
162,Cole Palmer,Manchester City,MID,8.002939,16.5,2,24.75
194,Chris Wood,Nottingham Forest,FWD,6.003879,6.0,7,6.00


##### Rank


In [117]:
fg_managers_33 = fg_managers[fg_managers['gameweek'] == 33]
fg_managers_33[['points']]

position_33 = (fg_managers_33['points'] > total_points_33).sum() + 1

# 4. Display result
total_managers_33 = len(fg_managers_33)
percentile_33 = percentile_34 = 100 * (1 - (position_33 - 1) / total_managers_33)

print(total_points_33, total_managers_33)

print(f"Your predicted player’s total of {total_points_33} points would rank:")
print(f"→ Position: {position_33} out of {total_managers_33} managers")
print(f"→ Percentile: Top {percentile_33:.2f}%")


# Optional: show nearby ranks for context
fg_sorted_33 = fg_managers_33.sort_values('points', ascending=False).reset_index(drop=True)

nearby_33 = fg_sorted_33.iloc[max(position_33-15, 0):position_33+2][['points','prize']]

player_prize_33 = fg_sorted_33.iloc[position_33 - 1]['prize']
if pd.isna(player_prize_33):
    player_prize_33 = "No prize"

print(f"→ Prize at position {position_33}: {player_prize_33}")
print("\nClosest scores around you:")
print(nearby_33)

88.95 246
Your predicted player’s total of 88.95 points would rank:
→ Position: 22 out of 246 managers
→ Percentile: Top 91.46%
→ Prize at position 22: R 21.95

Closest scores around you:
    points    prize
7    101.0  R 99.75
8     99.0  R 99.75
9     98.0  R 33.25
10    98.0  R 33.25
11    98.0  R 33.25
12    97.0  R 21.95
13    97.0  R 21.95
14    97.0  R 21.95
15    92.5  R 21.95
16    92.0  R 21.95
17    92.0  R 21.95
18    90.0  R 21.95
19    90.0  R 21.95
20    89.0  R 21.95
21    88.5  R 21.95
22    88.0  R 21.95
23    87.5  R 21.95


#### GW34


##### Select 11


In [86]:
players_34 = preds[preds['round'] == 34].copy()


players_34['player_id'] = list(range(len(players_34)))

players_34.rename(columns={'fpl_name': 'name', 'value': 'price'}, inplace=True)
players_34 = players_34.reset_index(drop=True)

selected_34 = run_pipeline(players_34)

Expert selected count (11-man heuristic): 11
IRL epoch 50/200 loss=-220.5320
IRL epoch 100/200 loss=-2399.8882
IRL epoch 150/200 loss=-10203.9346
IRL epoch 200/200 loss=-27924.3652
HGAT Policy Embeddings Shape: torch.Size([319, 32])

--- OPTIMAL FPL TEAM (IRL-HGAT-ILP RESULT) ---
                         name               team position  price   xP_pred  \
159           Kevin De Bruyne    Manchester City      MID   10.4  7.674042   
173    Bruno Borges Fernandes  Manchester United      MID    8.3  8.550670   
274               Jérémy Doku    Manchester City      MID    6.4  6.622347   
95               Eberechi Eze     Crystal Palace      MID    6.1  6.390789   
189            Anthony Gordon   Newcastle United      MID    6.1  6.925715   
24   Emiliano Martínez Romero        Aston Villa       GK    5.3  4.886585   
28              Ollie Watkins        Aston Villa      FWD    8.9  6.380586   
40            Dominic Solanke        Bournemouth      FWD    7.2  5.331156   
15             Be

##### Get Scores


In [88]:
# Get total points
players_selected_34 = players_34[players_34['fpl_id'].isin(selected_34['fpl_id'])].copy()
# players_selected_34 = players_selected_34.copy()
# 1. Create the rank column based on 'xp_preds'
# method='dense' ensures ties get the same rank, and the next rank is sequential
players_selected_34['xp_rank'] = players_selected_34['xP_pred'].rank(method='dense', ascending=False).astype(int)

# 2. Define the conditions and their corresponding multipliers
conditions = [
    players_selected_34['xp_rank'] == 1,  # Highest xp_preds
    players_selected_34['xp_rank'] == 2   # Second highest xp_preds
]
multipliers = [2.0, 1.5] # [multiplier for rank 1, multiplier for rank 2]

# 34 Use np.select() to apply the multiplier
# The 'default=1.0' means any rank other than 1 or 2 gets a 1.0 multiplier
players_selected_34['score'] = players_selected_34['xP'] * np.select(conditions, multipliers, default=1.0)
total_points_34 = sum(players_selected_34['score'].values)
print(total_points_34, players_selected_34['xP'].values, '--------->', players_selected_34['xP_pred'].values)

players_selected_34[['name', 'team', 'position', 'xP_pred', 'xP', 'xp_rank', 'score']]

111.39999999999999 [13.4 18.   5.1  6.5  8.1 11.   7.4 10.9  5.8  4.8  5.8] ---------> [5.43605941 5.16783334 4.88658533 6.38058621 5.33115595 6.39078907
 7.67404181 8.55066976 6.92571519 7.52763563 6.62234693]


,name,team,position,xP_pred,xP,xp_rank,score
10,William Saliba,Arsenal,DEF,5.436059,13.4,8,13.4
15,Benjamin White,Arsenal,DEF,5.167833,18.0,10,18.0
24,Emiliano Martínez Romero,Aston Villa,GK,4.886585,5.1,11,5.1
28,Ollie Watkins,Aston Villa,FWD,6.380586,6.5,7,6.5
40,Dominic Solanke,Bournemouth,FWD,5.331156,8.1,9,8.1
95,Eberechi Eze,Crystal Palace,MID,6.390789,11.0,6,11.0
159,Kevin De Bruyne,Manchester City,MID,7.674042,7.4,2,11.1
173,Bruno Borges Fernandes,Manchester United,MID,8.550670,10.9,1,21.8
189,Anthony Gordon,Newcastle United,MID,6.925715,5.8,4,5.8
195,Fabian Schär,Newcastle United,DEF,7.527636,4.8,3,4.8


##### Rank


In [118]:
fg_managers_34 = fg_managers[fg_managers['gameweek'] == 34]
fg_managers_34[['points']]

position_34 = (fg_managers_34['points'] > total_points_34).sum() + 1

# 4. Display result
total_managers_34 = len(fg_managers_34)
percentile_34 = percentile_34 = 100 * (1 - (position_34 - 1) / total_managers_34)

print(total_points_34, total_managers_34)

print(f"Your predicted player’s total of {total_points_34} points would rank:")
print(f"→ Position: {position_34} out of {total_managers_34} managers")
print(f"→ Percentile: Top {percentile_34:.2f}%")


# Optional: show nearby ranks for context
fg_sorted_34 = fg_managers_34.sort_values('points', ascending=False).reset_index(drop=True)

nearby_34 = fg_sorted_34.iloc[max(position_34-15, 0):position_34+2][['points','prize']]

player_prize_34 = fg_sorted_34.iloc[position_34 - 1]['prize']
if pd.isna(player_prize_34):
    player_prize_34 = "No prize"

print(f"→ Prize at position {position_34}: {player_prize_34}")
print("\nClosest scores around you:")
print(nearby_34)

111.39999999999999 203
Your predicted player’s total of 111.39999999999999 points would rank:
→ Position: 108 out of 203 managers
→ Percentile: Top 47.29%
→ Prize at position 108: No prize

Closest scores around you:
     points prize
93    117.5   NaN
94    117.5   NaN
95    116.5   NaN
96    116.5   NaN
97    116.5   NaN
98    116.5   NaN
99    116.0   NaN
100   115.5   NaN
101   114.5   NaN
102   112.5   NaN
103   112.5   NaN
104   112.5   NaN
105   111.5   NaN
106   111.5   NaN
107   110.0   NaN
108   109.5   NaN
109   109.5   NaN


#### GW35


##### Select 11


In [90]:
players_35 = preds[preds['round'] == 35].copy()


players_35['player_id'] = list(range(len(players_35)))

players_35.rename(columns={'fpl_name': 'name', 'value': 'price'}, inplace=True)
players_35 = players_35.reset_index(drop=True)

selected_35 = run_pipeline(players_35)

Expert selected count (11-man heuristic): 11
IRL epoch 50/200 loss=-208.3313
IRL epoch 100/200 loss=-2263.2744
IRL epoch 150/200 loss=-9606.4521
IRL epoch 200/200 loss=-26274.8867
HGAT Policy Embeddings Shape: torch.Size([323, 32])

--- OPTIMAL FPL TEAM (IRL-HGAT-ILP RESULT) ---
                       name               team position  price   xP_pred  \
149         Kevin De Bruyne    Manchester City      MID   10.4  9.646874   
4           Martin Ødegaard            Arsenal      MID    8.6  7.436537   
160  Bruno Borges Fernandes  Manchester United      MID    8.4  9.573815   
1               Kai Havertz            Arsenal      MID    7.4  8.054727   
155             Cole Palmer    Manchester City      MID    6.2  8.180694   
179         Martin Dubravka   Newcastle United       GK    4.3  5.094731   
26            Ollie Watkins        Aston Villa      FWD    9.0  8.792506   
94     Jean-Philippe Mateta     Crystal Palace      FWD    5.1  7.595729   
12           Benjamin White         

##### Get Scores


In [91]:
# Get total points
players_selected_35 = players_35[players_35['fpl_id'].isin(selected_35['fpl_id'])].copy()
# players_selected_35 = players_selected_35.copy()
# 1. Create the rank column based on 'xp_preds'
# method='dense' ensures ties get the same rank, and the next rank is sequential
players_selected_35['xp_rank'] = players_selected_35['xP_pred'].rank(method='dense', ascending=False).astype(int)

# 2. Define the conditions and their corresponding multipliers
conditions = [
    players_selected_35['xp_rank'] == 1,  # Highest xp_preds
    players_selected_35['xp_rank'] == 2   # Second highest xp_preds
]
multipliers = [2.0, 1.5] # [multiplier for rank 1, multiplier for rank 2]

# 3. Use np.select() to apply the multiplier
# The 'default=1.0' means any rank other than 1 or 2 gets a 1.0 multiplier
players_selected_35['score'] = players_selected_35['xP'] * np.select(conditions, multipliers, default=1.0)
total_points_35 = sum(players_selected_35['score'].values)
print(total_points_35, players_selected_35['xP'].values, '--------->', players_selected_35['xP_pred'].values)

players_selected_35[['name', 'team', 'position', 'xP_pred', 'xP', 'xp_rank', 'score']]

110.85000000000001 [ 8.8  7.   7.7  6.7  7.6  9.  18.6 11.3  5.5  6.   8. ] ---------> [8.05472687 7.43653743 7.51450522 8.792506   7.59572854 9.64687404
 8.18069417 9.57381497 5.09473083 7.211556   7.45549996]


,name,team,position,xP_pred,xP,xp_rank,score
1,Kai Havertz,Arsenal,MID,8.054727,8.8,5,8.80
4,Martin Ødegaard,Arsenal,MID,7.436537,7.0,9,7.00
12,Benjamin White,Arsenal,DEF,7.514505,7.7,7,7.70
26,Ollie Watkins,Aston Villa,FWD,8.792506,6.7,3,6.70
94,Jean-Philippe Mateta,Crystal Palace,FWD,7.595729,7.6,6,7.60
149,Kevin De Bruyne,Manchester City,MID,9.646874,9.0,1,18.00
155,Cole Palmer,Manchester City,MID,8.180694,18.6,4,18.60
160,Bruno Borges Fernandes,Manchester United,MID,9.573815,11.3,2,16.95
179,Martin Dubravka,Newcastle United,GK,5.094731,5.5,11,5.50
187,Fabian Schär,Newcastle United,DEF,7.211556,6.0,10,6.00


##### Rank


In [119]:
fg_managers_35 = fg_managers[fg_managers['gameweek'] == 35]
fg_managers_35[['points']]

position_35 = (fg_managers_35['points'] > total_points_35).sum() + 1

# 4. Display result
total_managers_35 = len(fg_managers_35)
percentile_35 = percentile_34 = 100 * (1 - (position_35 - 1) / total_managers_35)

print(total_points_35, total_managers_35)

print(f"Your predicted player’s total of {total_points_35} points would rank:")
print(f"→ Position: {position_35} out of {total_managers_35} managers")
print(f"→ Percentile: Top {percentile_35:.2f}%")


# Optional: show nearby ranks for context
fg_sorted_35 = fg_managers_35.sort_values('points', ascending=False).reset_index(drop=True)

nearby_35 = fg_sorted_35.iloc[max(position_35-15, 0):position_35+2][['points','prize']]

player_prize_35 = fg_sorted_35.iloc[position_35 - 1]['prize']
if pd.isna(player_prize_35):
    player_prize_35 = "No prize"

print(f"→ Prize at position {position_35}: {player_prize_35}")
print("\nClosest scores around you:")
print(nearby_35)

110.85000000000001 207
Your predicted player’s total of 110.85000000000001 points would rank:
→ Position: 2 out of 207 managers
→ Percentile: Top 99.52%
→ Prize at position 2: R 1,695.75

Closest scores around you:
   points       prize
0   111.5  R 3,890.25
1   100.0  R 1,695.75
2    97.5  R 1,097.25
3    94.5    R 522.50


#### GW36


##### Select 11


In [93]:
players_36 = preds[preds['round'] == 36].copy()


players_36['player_id'] = list(range(len(players_36)))

players_36.rename(columns={'fpl_name': 'name', 'value': 'price'}, inplace=True)
players_36 = players_36.reset_index(drop=True)

selected_36 = run_pipeline(players_36)

Expert selected count (11-man heuristic): 11
IRL epoch 50/200 loss=-320.8398
IRL epoch 100/200 loss=-3700.4075
IRL epoch 150/200 loss=-15996.7646
IRL epoch 200/200 loss=-44068.1094
HGAT Policy Embeddings Shape: torch.Size([324, 32])

--- OPTIMAL FPL TEAM (IRL-HGAT-ILP RESULT) ---
                name              team position  price    xP_pred  Is_Captain
152  Kevin De Bruyne   Manchester City      MID   10.5  10.348960        True
6        Bukayo Saka           Arsenal      MID    9.0   9.876890       False
1        Kai Havertz           Arsenal      MID    7.5   8.632476       False
180   Anthony Gordon  Newcastle United      MID    6.3   8.321610       False
113  Jordan Pickford           Everton       GK    4.8   6.701558       False
156   Erling Haaland   Manchester City      FWD   14.1   7.828144       False
24     Ollie Watkins       Aston Villa      FWD    9.0   6.381827       False
181   Alexander Isak  Newcastle United      FWD    8.3   7.335730       False
12    Benjamin Wh

##### Get Scores


In [94]:
# Get total points
players_selected_36 = players_36[players_36['fpl_id'].isin(selected_36['fpl_id'])].copy()
# players_selected_36 = players_selected_36.copy()
# 1. Create the rank column based on 'xp_preds'
# method='dense' ensures ties get the same rank, and the next rank is sequential
players_selected_36['xp_rank'] = players_selected_36['xP_pred'].rank(method='dense', ascending=False).astype(int)

# 2. Define the conditions and their corresponding multipliers
conditions = [
    players_selected_36['xp_rank'] == 1,  # Highest xp_preds
    players_selected_36['xp_rank'] == 2   # Second highest xp_preds
]
multipliers = [2.0, 1.5] # [multiplier for rank 1, multiplier for rank 2]

# 3. Use np.select() to apply the multiplier
# The 'default=1.0' means any rank other than 1 or 2 gets a 1.0 multiplier
players_selected_36['score'] = players_selected_36['xP'] * np.select(conditions, multipliers, default=1.0)
total_points_36 = sum(players_selected_36['score'].values)
print(total_points_36, players_selected_36['xP'].values, '--------->', players_selected_36['xP_pred'].values)

players_selected_36[['name', 'team', 'position', 'xP_pred', 'xP', 'xp_rank', 'score']]

101.30000000000001 [ 8.2  7.6  8.6  5.5  4.6  6.7  7.  10.   9.   9.  10.8] ---------> [ 8.6324761   9.87688973  8.47279522  6.38182739  7.06767036  6.70155766
 10.34896042  7.82814365  8.32161007  7.3357297  10.36621079]


,name,team,position,xP_pred,xP,xp_rank,score
1,Kai Havertz,Arsenal,MID,8.632476,8.2,4,8.2
6,Bukayo Saka,Arsenal,MID,9.876890,7.6,3,7.6
12,Benjamin White,Arsenal,DEF,8.472795,8.6,5,8.6
24,Ollie Watkins,Aston Villa,FWD,6.381827,5.5,11,5.5
31,Marcos Senesi,Bournemouth,DEF,7.067670,4.6,9,4.6
113,Jordan Pickford,Everton,GK,6.701558,6.7,10,6.7
152,Kevin De Bruyne,Manchester City,MID,10.348960,7.0,2,10.5
156,Erling Haaland,Manchester City,FWD,7.828144,10.0,7,10.0
180,Anthony Gordon,Newcastle United,MID,8.321610,9.0,6,9.0
181,Alexander Isak,Newcastle United,FWD,7.335730,9.0,8,9.0


##### Rank


In [120]:
fg_managers_36 = fg_managers[fg_managers['gameweek'] == 36]
fg_managers_36[['points']]

position_36 = (fg_managers_36['points'] > total_points_36).sum() + 1

# 4. Display result
total_managers_36 = len(fg_managers_36)
percentile_36 = percentile_34 = 100 * (1 - (position_36 - 1) / total_managers_36)

print(total_points_36, total_managers_36)

print(f"Your predicted player’s total of {total_points_36} points would rank:")
print(f"→ Position: {position_36} out of {total_managers_36} managers")
print(f"→ Percentile: Top {percentile_36:.2f}%")


# Optional: show nearby ranks for context
fg_sorted_36 = fg_managers_36.sort_values('points', ascending=False).reset_index(drop=True)

nearby_36 = fg_sorted_36.iloc[max(position_36-15, 0):position_36+2][['points','prize']]

player_prize_36 = fg_sorted_36.iloc[position_36 - 1]['prize']
if pd.isna(player_prize_36):
    player_prize_36 = "No prize"

print(f"→ Prize at position {position_36}: {player_prize_36}")
print("\nClosest scores around you:")
print(nearby_36)

101.30000000000001 229
Your predicted player’s total of 101.30000000000001 points would rank:
→ Position: 39 out of 229 managers
→ Percentile: Top 83.41%
→ Prize at position 39: R 22.39

Closest scores around you:
    points    prize
24   104.0  R 22.39
25   103.5  R 22.39
26   103.5  R 22.39
27   103.5  R 22.39
28   103.5  R 22.39
29   103.5  R 22.39
30   102.5  R 22.39
31   102.0  R 22.39
32   102.0  R 22.39
33   102.0  R 22.39
34   102.0  R 22.39
35   102.0  R 22.39
36   101.5  R 22.39
37   101.5  R 22.39
38   101.0  R 22.39
39   100.0  R 22.39
40    99.0  R 22.39


#### GW37


##### Select 11


In [96]:
players_37 = preds[preds['round'] == 37].copy()


players_37['player_id'] = list(range(len(players_37)))

players_37.rename(columns={'fpl_name': 'name', 'value': 'price'}, inplace=True)
players_37 = players_37.reset_index(drop=True)

selected_37 = run_pipeline(players_37)

Expert selected count (11-man heuristic): 11
IRL epoch 50/200 loss=-245.1174
IRL epoch 100/200 loss=-2983.7952
IRL epoch 150/200 loss=-13097.8633
IRL epoch 200/200 loss=-36324.8945
HGAT Policy Embeddings Shape: torch.Size([344, 32])

--- OPTIMAL FPL TEAM (IRL-HGAT-ILP RESULT) ---
                       name               team position  price    xP_pred  \
160         Kevin De Bruyne    Manchester City      MID   10.6   7.405214   
8               Bukayo Saka            Arsenal      MID    8.9   9.297862   
174  Bruno Borges Fernandes  Manchester United      MID    8.5   6.669539   
1               Kai Havertz            Arsenal      MID    7.5   6.872515   
106           Michael Olise     Crystal Palace      MID    5.6   7.824444   
293          Đorđe Petrović            Chelsea       GK    4.7   5.044118   
165          Erling Haaland    Manchester City      FWD   14.2  10.799903   
93          Nicolas Jackson            Chelsea      FWD    7.0   7.188019   
271          Joško Gvardio

##### Get Scores


In [97]:
# Get total points
players_selected_37 = players_37[players_37['fpl_id'].isin(selected_37['fpl_id'])].copy()
# players_selected_37 = players_selected_37.copy()
# 1. Create the rank column based on 'xp_preds'
# method='dense' ensures ties get the same rank, and the next rank is sequential
players_selected_37['xp_rank'] = players_selected_37['xP_pred'].rank(method='dense', ascending=False).astype(int)

# 2. Define the conditions and their corresponding multipliers
conditions = [
    players_selected_37['xp_rank'] == 1,  # Highest xp_preds
    players_selected_37['xp_rank'] == 2   # Second highest xp_preds
]
multipliers = [2.0, 1.5] # [multiplier for rank 1, multiplier for rank 2]

# 3. Use np.select() to apply the multiplier
# The 'default=1.0' means any rank other than 1 or 2 gets a 1.0 multiplier
players_selected_37['score'] = players_selected_37['xP'] * np.select(conditions, multipliers, default=1.0)
total_points_37 = sum(players_selected_37['score'].values)
print(total_points_37, players_selected_37['xP'].values, '--------->', players_selected_37['xP_pred'].values)

players_selected_37[['name', 'team', 'position', 'xP_pred', 'xP', 'xp_rank', 'score']]

138.45000000000002 [ 9.2  5.7 11.   4.4  8.6  7.9 15.1 17.9  7.4 23.1  7.4] ---------> [ 6.87251482  9.29786187  7.18801906  5.731815    7.82444444  5.54862572
  7.40521356 10.79990285  6.66953872  8.93402928  5.04411829]


,name,team,position,xP_pred,xP,xp_rank,score
1,Kai Havertz,Arsenal,MID,6.872515,9.2,7,9.20
8,Bukayo Saka,Arsenal,MID,9.297862,5.7,2,8.55
93,Nicolas Jackson,Chelsea,FWD,7.188019,11.0,6,11.00
105,Tyrick Mitchell,Crystal Palace,DEF,5.731815,4.4,9,4.40
106,Michael Olise,Crystal Palace,MID,7.824444,8.6,4,8.60
111,Jarrad Branthwaite,Everton,DEF,5.548626,7.9,10,7.90
160,Kevin De Bruyne,Manchester City,MID,7.405214,15.1,5,15.10
165,Erling Haaland,Manchester City,FWD,10.799903,17.9,1,35.80
174,Bruno Borges Fernandes,Manchester United,MID,6.669539,7.4,8,7.40
271,Joško Gvardiol,Manchester City,DEF,8.934029,23.1,3,23.10


##### Rank


In [121]:
fg_managers_37 = fg_managers[fg_managers['gameweek'] == 37]
fg_managers_37[['points']]

position_37 = (fg_managers_37['points'] > total_points_37).sum() + 1

# 4. Display result
total_managers_37 = len(fg_managers_37)
percentile_37 = percentile_34 = 100 * (1 - (position_37 - 1) / total_managers_37)

print(total_points_37, total_managers_37)

print(f"Your predicted player’s total of {total_points_37} points would rank:")
print(f"→ Position: {position_37} out of {total_managers_37} managers")
print(f"→ Percentile: Top {percentile_37:.2f}%")


# Optional: show nearby ranks for context
fg_sorted_37 = fg_managers_37.sort_values('points', ascending=False).reset_index(drop=True)

nearby_37 = fg_sorted_37.iloc[max(position_37-15, 0):position_37+2][['points','prize']]

player_prize_37 = fg_sorted_37.iloc[position_37 - 1]['prize']
if pd.isna(player_prize_37):
    player_prize_37 = "No prize"

print(f"→ Prize at position {position_37}: {player_prize_37}")
print("\nClosest scores around you:")
print(nearby_37)

138.45000000000002 243
Your predicted player’s total of 138.45000000000002 points would rank:
→ Position: 2 out of 243 managers
→ Percentile: Top 99.59%
→ Prize at position 2: R 1,695.75

Closest scores around you:
   points       prize
0   140.0  R 3,890.25
1   138.0  R 1,695.75
2   136.5  R 1,097.25
3   133.0    R 460.75


#### GW38


##### Select 11


In [99]:
players_38 = preds[preds['round'] == 38].copy()


players_38['player_id'] = list(range(len(players_38)))

players_38.rename(columns={'fpl_name': 'name', 'value': 'price'}, inplace=True)
players_38 = players_38.reset_index(drop=True)

selected_38 = run_pipeline(players_38)

Expert selected count (11-man heuristic): 11
IRL epoch 50/200 loss=-270.6257
IRL epoch 100/200 loss=-3061.3894
IRL epoch 150/200 loss=-13222.8672
IRL epoch 200/200 loss=-36432.4961
HGAT Policy Embeddings Shape: torch.Size([371, 32])

--- OPTIMAL FPL TEAM (IRL-HGAT-ILP RESULT) ---
                   name             team position  price    xP_pred  \
156     Kevin De Bruyne  Manchester City      MID   10.6   8.333934   
1           Kai Havertz          Arsenal      MID    7.6   8.466716   
11     Leandro Trossard          Arsenal      MID    6.6   8.208125   
104       Michael Olise   Crystal Palace      MID    5.7   7.945739   
88         Noni Madueke          Chelsea      MID    5.3   7.852796   
47    David Raya Martin          Arsenal       GK    5.3   6.146125   
160      Erling Haaland  Manchester City      FWD   14.3   9.395076   
136          Cody Gakpo        Liverpool      FWD    7.1   6.699286   
140    Andrew Robertson        Liverpool      DEF    6.5   7.274200   
284      

##### Get Scores


In [100]:
# Get total points
players_selected_38 = players_38[players_38['fpl_id'].isin(selected_38['fpl_id'])].copy()
# players_selected_38 = players_selected_38.copy()
# 1. Create the rank column based on 'xp_preds'
# method='dense' ensures ties get the same rank, and the next rank is sequential
players_selected_38['xp_rank'] = players_selected_38['xP_pred'].rank(method='dense', ascending=False).astype(int)

# 2. Define the conditions and their corresponding multipliers
conditions = [
    players_selected_38['xp_rank'] == 1,  # Highest xp_preds
    players_selected_38['xp_rank'] == 2   # Second highest xp_preds
]
multipliers = [2.0, 1.5] # [multiplier for rank 1, multiplier for rank 2]

# 3. Use np.select() to apply the multiplier
# The 'default=1.0' means any rank other than 1 or 2 gets a 1.0 multiplier
players_selected_38['score'] = players_selected_38['xP'] * np.select(conditions, multipliers, default=1.0)
total_points_38 = sum(players_selected_38['score'].values)
print(total_points_38, players_selected_38['xP'].values, '--------->', players_selected_38['xP_pred'].values)

players_selected_38[['name', 'team', 'position', 'xP_pred', 'xP', 'xp_rank', 'score']]

87.80000000000001 [ 8.5  7.   4.5  5.7  8.5  3.5  6.   6.   5.2  8.2 10.3] ---------> [ 8.46671624  8.208125    6.14612485  7.85279648  7.94573883  4.78905057
  6.69928604  7.27420006  8.33393421  9.39507631 10.31084106]


,name,team,position,xP_pred,xP,xp_rank,score
1,Kai Havertz,Arsenal,MID,8.466716,8.5,3,8.5
11,Leandro Trossard,Arsenal,MID,8.208125,7.0,5,7.0
47,David Raya Martin,Arsenal,GK,6.146125,4.5,10,4.5
88,Noni Madueke,Chelsea,MID,7.852796,5.7,7,5.7
104,Michael Olise,Crystal Palace,MID,7.945739,8.5,6,8.5
108,Jarrad Branthwaite,Everton,DEF,4.789051,3.5,11,3.5
136,Cody Gakpo,Liverpool,FWD,6.699286,6.0,9,6.0
140,Andrew Robertson,Liverpool,DEF,7.274200,6.0,8,6.0
156,Kevin De Bruyne,Manchester City,MID,8.333934,5.2,4,5.2
160,Erling Haaland,Manchester City,FWD,9.395076,8.2,2,12.3


##### Rank


In [122]:
fg_managers_38 = fg_managers[fg_managers['gameweek'] == 38]
fg_managers_38[['points']]

position_38 = (fg_managers_38['points'] > total_points_38).sum() + 1

# 4. Display result
total_managers_38 = len(fg_managers_38)
percentile_38 = percentile_34 = 100 * (1 - (position_38 - 1) / total_managers_38)

print(total_points_38, total_managers_38)

print(f"Your predicted player’s total of {total_points_38} points would rank:")
print(f"→ Position: {position_38} out of {total_managers_38} managers")
print(f"→ Percentile: Top {percentile_38:.2f}%")


# Optional: show nearby ranks for context
fg_sorted_38 = fg_managers_38.sort_values('points', ascending=False).reset_index(drop=True)

nearby_38 = fg_sorted_38.iloc[max(position_38-15, 0):position_38+2][['points','prize']]

player_prize_38 = fg_sorted_38.iloc[position_38 - 1]['prize']
if pd.isna(player_prize_38):
    player_prize_38 = "No prize"

print(f"→ Prize at position {position_38}: {player_prize_38}")
print("\nClosest scores around you:")
print(nearby_38)

87.80000000000001 256
Your predicted player’s total of 87.80000000000001 points would rank:
→ Position: 33 out of 256 managers
→ Percentile: Top 87.50%
→ Prize at position 33: R 20.32

Closest scores around you:
    points    prize
18    94.0  R 20.32
19    93.5  R 20.32
20    93.5  R 20.32
21    92.5  R 20.32
22    92.0  R 20.32
23    92.0  R 20.32
24    91.5  R 20.32
25    91.0  R 20.32
26    91.0  R 20.32
27    91.0  R 20.32
28    91.0  R 20.32
29    90.5  R 20.32
30    90.5  R 20.32
31    89.0  R 20.32
32    87.5  R 20.32
33    87.5  R 20.32
34    86.0  R 20.32


#### Results


In [124]:
gameweeks = range(31, 39)

results = {
    'gameweek': list(gameweeks),
    'score': [globals()[f'total_points_{gw}'] for gw in gameweeks],
    'position': [globals()[f'position_{gw}'] for gw in gameweeks],
    'total_managers': [globals()[f'total_managers_{gw}'] for gw in gameweeks],
    'percentile': [globals()[f'percentile_{gw}'] for gw in gameweeks],
    'prize': [globals()[f'player_prize_{gw}'] for gw in gameweeks],
    'players': [globals().get(f'players_selected_{gw}', pd.DataFrame(columns=['name']))['name'].tolist() for gw in gameweeks]
}
pd.DataFrame(results)


,gameweek,score,position,total_managers,percentile,prize,players
0,31,94.05,1,217,100.000000,"R 3,890.25","[William Saliba, Benjamin White, Ezri Konsa Ng..."
1,32,86.25,8,234,97.008547,R 99.75,"[Martin Ødegaard, William Saliba, Benjamin Whi..."
2,33,88.95,22,246,91.463415,R 21.95,"[Kai Havertz, Benjamin White, David Raya Marti..."
3,34,111.40,108,203,87.500000,No prize,"[William Saliba, Benjamin White, Emiliano Mart..."
4,35,110.85,2,207,99.516908,"R 1,695.75","[Kai Havertz, Martin Ødegaard, Benjamin White,..."
5,36,101.30,39,229,83.406114,R 22.39,"[Kai Havertz, Bukayo Saka, Benjamin White, Oll..."
6,37,138.45,2,243,99.588477,"R 1,695.75","[Kai Havertz, Bukayo Saka, Nicolas Jackson, Ty..."
7,38,87.80,33,256,87.500000,R 20.32,"[Kai Havertz, Leandro Trossard, David Raya Mar..."
